# অলীকবচন — final pipeline

**Ground-truth matcher (5 sources) + gemma-4 judge + SummaC NLI + math solver + hidden-state probing**, fused by an OOF-validated decision layer.

Current Kaggle-confirmed best: **0.843** (his + matcher + logreg @ thr 0.52).

New here: **hidden-state probing** — the judge model's internal activations read into the decision layer. This is the only signal that does not depend on the matcher finding a public answer key, so it is the Phase-2 fallback if matcher coverage craters on fresh data.

**Judge backend** is `CFG['model_backend']`: `"12b"` (default, the 0.843 config) or `"31b-bnb"` (gemma-4-31B bf16 mount + bitsandbytes 4-bit across 2×T4). GGUF is deliberately not offered — llama.cpp cannot expose intermediate activations, which would disable probing.

**Needs:** GPU (2×T4 or P100), the gemma-4 Kaggle Model attached, and the mDeBERTa multilingual-NLI dataset. Internet ON for Phase 1 (the matcher auto-downloads its 5 QA sources); for Phase 2 pre-bundle those parquet caches.

## ⚠️ Phase 2 (offline) — required attached inputs

Phase 1 (internet ON) needs nothing attached; everything downloads at runtime.
For Phase 2 every network fetch below must instead be satisfied by an attached Kaggle input.
The code checks `/kaggle/input/**` **first** in every case, so attaching these is sufficient:

| # | Attach as input | Satisfies |
|---|---|---|
| 1 | Dataset with the 5 matcher caches: `nctb_qa_train.parquet`, `tydiqa_goldp_bengali.parquet`, `indicqa_bn.json`, `banglarqa_train.json`, `bangla_mmlu.parquet` | Ground-truth matcher (the 0.843 lever) — **without this it silently degrades to neutral features** |
| 2 | Dataset with a `wheelhouse/` folder of wheels: `transformers` (gemma-4-capable, ≥5.14.1), `accelerate`, `bitsandbytes`, `sentence-transformers`, `xgboost`, `kagglehub` | Both pip cells (Part 0 + Part 0.5 bootstrap) |
| 3 | gemma-4 weights (Kaggle Model mount or dataset copy, dir name matching `*gemma*4*31b*` / `*gemma*4*12b*`) | Judge + hidden-state probe |
| 4 | `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` model files (dir containing `config.json`, name containing `mdeberta`) | SummaC NLI lane |
| 5 | `facebook/nllb-200-distilled-600M` model files (dir name containing `nllb`) | Cross-lingual translation lane |

Graceful-degradation order if something is missing: matcher → neutral features (biggest score
loss); NLLB → cross-lingual features fall back; judge/NLI missing → pipeline cannot run.


In [ ]:
# Phase 1 (internet ON): plain PyPI install. Phase 2 (offline): attach a Kaggle
# Dataset containing a "wheelhouse" folder of pre-downloaded wheels and this installs
# strictly from it -- no network attempted.
import glob, subprocess, sys
_wh = glob.glob("/kaggle/input/**/wheelhouse", recursive=True)
_pkgs = ["accelerate", "sentence-transformers", "xgboost", "bitsandbytes", "kagglehub"]
_cmd = [sys.executable, "-m", "pip", "install", "-q"]
if _wh:
    print("wheelhouse found:", _wh[0], "-- installing OFFLINE from bundled wheels")
    _cmd += ["--no-index", "--find-links", _wh[0]]
_r = subprocess.run(_cmd + _pkgs, capture_output=True, text=True)
if _r.returncode != 0:
    print("pip failed (continuing -- Kaggle base image provides most of these):")
    print(_r.stderr[-800:])
else:
    print("deps installed", "(offline)" if _wh else "(online)")


## Part 0.5 — transformers bootstrap (MUST run before any transformers import)

In [ ]:
"""
Fusion pipeline: combines two independently-built approaches.

  Lane 1 (teammate's, adapted near-verbatim from phase2-final.ipynb):
    - SummaC-ZS style windowed NLI via MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
    - gemma-4-12b-it bilingual (bn + en) judge, logit-based Yes/No scoring
    - 15-template deterministic math-word-problem solver

  Lane 2 (ours): translation-based cross-lingual NLI (NLLB bn->en + NLI re-check),
    joggota deterministic rules (idiom/spelling/math, now bug-fixed), response-quality
    heuristics, and the two cheap lexical features that carried the most weight in our
    real Kaggle run: context_containment, novel_char_ratio.

  Then: proper StratifiedKFold OOF comparison of (his features alone / ours alone /
  combined) x (GaussianNB / LogisticRegression / XGBoost), reporting hallu_F1 honestly
  (no in-sample leakage, no double-dipped threshold search) before picking a final model.

Run this on Kaggle (needs GPU for gemma-4 + NLI + NLLB). Mirrors phase2-final.ipynb's
offline plumbing (wheelhouse-based transformers upgrade, Kaggle Model registry for gemma-4).
"""
import os
import re
import gc
import glob
import json
import random
import warnings
import subprocess
import sys

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# Local modules (already Phase-2-compliant, no internet needed)
# [inlined above]
# [inlined above]
# [inlined above]
# [inlined above]

# ----------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------
CFG = {
    "seed": 42,
    "llm_batch": 8,
    "llm_token_budget": 12288,
    "nli_batch": 64,
    "win_list": [2, 3],
    "max_ctx_chars": 3000, "max_prompt_chars": 1500, "max_resp_chars": 2500,
    "max_len_llm": 2048,

    # ---- judge backend ----------------------------------------------------
    # "12b"     -> google/gemma-4/transformers/gemma-4-12b-it  (the 0.843 config)
    # "31b-bnb" -> gemma-4-31B-it bf16 mount, bitsandbytes 4-bit, device_map="auto"
    #              across 2xT4 (~15-18GB of ~30GB). Slower, but hidden states available.
    #
    # NOT offered: "31b-gguf" (llama.cpp). GGUF quants are real and ungated
    # (unsloth/gemma-4-31B-it-qat-GGUF, Q4_K_XL @ 17.3GB fits 2xT4 fine) and it is the
    # FASTER path for pure judging -- but llama.cpp does not expose intermediate-layer
    # activations, so choosing it forecloses hidden-state probing entirely. Since probing
    # is the cheaper test of the same hypothesis, transformers+bnb is the right default.
    "model_backend": "31b-bnb",
    
    
    "force_model_path": None,

    # ---- hidden-state probing --------------------------------------------
    "probe_hidden_states": True,
    "probe_fractions": (0.25, 0.50, 0.75, 0.95),  # depth fractions -> layer indices

    # ---- smoke test -------------------------------------------------------
    # None = full run. An int caps the TEST set to that many rows for a fast
    # end-to-end inference dry-run; train stays full so the decision-layer CV
    # is unaffected. e.g. set 12 to sanity-check the whole pipeline in minutes.
    "smoke_n": None,
}

# Candidate Kaggle Model handles, tried in order. Upstream HF capitalizes the 31B as
# "gemma-4-31B-it" while the 12B is "gemma-4-12b-it", and Kaggle's variation slug doesn't
# always match HF's casing -- so try both rather than 404 after you've already waited.
MODEL_HANDLES = {
    "12b": [
        "google/gemma-4/transformers/gemma-4-12b-it",
    ],
    "31b-bnb": [
        "google/gemma-4/transformers/gemma-4-31b-it",
        "google/gemma-4/transformers/gemma-4-31B-it",
        "google/gemma-4/transformers/31b-it",
    ],
}


# HuggingFace fallback. The Kaggle Model registry and HF are SEPARATE -- a model being on
# HF (confirmed: google/gemma-4-31B-it is public and ungated) does not mean it is published
# as a Kaggle Model. If no Kaggle handle resolves, fall back to the HF repo id, which
# from_pretrained accepts directly. Phase 1 permits internet so this is fine; for Phase 2
# the weights must be pre-attached instead.
HF_FALLBACK = {
    "12b": "google/gemma-4-12b-it",
    # Unsloth's PRE-QUANTIZED bnb-4bit build, verified: 19.1 GB vs ~61 GB for bf16,
    # ungated, quant_method=bitsandbytes / nf4 / bf16 compute. Transformers-native, so
    # hidden-state probing still works -- which the GGUF builds would have foreclosed.
    # Downloading the bf16 and quantizing on the fly wastes ~42 GB for the same result.
    "31b-bnb": "unsloth/gemma-4-31B-it-unsloth-bnb-4bit",
}


# Local mounts are checked FIRST: an attached /kaggle/input path needs no download, and is
# the only option that works with Internet OFF (i.e. Phase 2). Community mirrors show up
# here, e.g. /kaggle/input/notebooks/danielhanchen/gemma4-31b-unsloth -- Unsloth ships
# PRE-QUANTIZED 4-bit builds (~18GB rather than the 61GB bf16), which also load much faster.
LOCAL_PATTERNS = {
    "12b": ["*gemma*4*12b*", "*gemma4*12b*"],
    "31b-bnb": ["*gemma*4*31b*", "*gemma4*31b*", "*gemma*4*31B*"],
}


def find_local_model(backend):
    """Search /kaggle/input for a directory holding a real base model.

    Deliberately rejects PEFT/LoRA adapter directories. A mount whose name looks perfect
    can be an adapter, not a model -- e.g. the community
    /kaggle/input/notebooks/danielhanchen/gemma4-31b-unsloth path is
    `library_name: peft`, tags lora/sft, i.e. someone's fine-tune output. It ships
    `adapter_config.json` + `adapter_model.safetensors` (a few hundred MB), NOT the 31B
    weights. Loading it as a base model fails; applying it as an adapter would silently
    alter judge behaviour with unknown SFT data, which is worse.
    """
    seen = []
    for pat in LOCAL_PATTERNS.get(backend, []):
        for cfg in glob.glob(f"/kaggle/input/**/{pat}/**/config.json", recursive=True) + \
                   glob.glob(f"/kaggle/input/**/{pat}/config.json", recursive=True):
            d = os.path.dirname(cfg)
            if d in seen:
                continue
            seen.append(d)
            if os.path.exists(os.path.join(d, "adapter_config.json")):
                print(f"  [skip] {d} is a PEFT/LoRA adapter, not a base model")
                continue
            try:
                with open(cfg, encoding="utf-8") as f:
                    meta = json.load(f)
                if not (meta.get("model_type") or meta.get("architectures")):
                    continue          # not a real model config
            except Exception:
                continue
            shards = [p for p in glob.glob(os.path.join(d, "*.safetensors"))
                      if "adapter" not in os.path.basename(p).lower()]
            if shards or glob.glob(os.path.join(d, "*.bin")):
                return d
    return None


def is_prequantized(path):
    """True if the checkpoint is already quantized (Unsloth/bnb builds are).

    Passing our own BitsAndBytesConfig on top of an already-quantized checkpoint is a
    conflict -- transformers should read the config's own quantization settings instead.
    """
    cfg = os.path.join(str(path), "config.json")
    if not os.path.exists(cfg):
        return False
    try:
        with open(cfg, encoding="utf-8") as f:
            return "quantization_config" in json.load(f)
    except Exception:
        return False


def resolve_model_path(backend):
    """Local mount -> Kaggle Model handles -> HuggingFace repo id."""
    import kagglehub
    errors = []
    
    forced = CFG.get("force_model_path")
    if forced:
        print(f"  using override: {forced}")
        return forced


    local = find_local_model(backend)
    if local:
        tag = "pre-quantized" if is_prequantized(local) else "full precision"
        print(f"  using LOCAL mount ({tag}): {local}")
        return local

    for handle in MODEL_HANDLES[backend]:
        try:
            path = kagglehub.model_download(handle)
            print(f"  resolved '{handle}' -> {path}")
            return path
        except Exception as e:
            errors.append(f"    kaggle:{handle}: {type(e).__name__}")
            print(f"  '{handle}' unavailable, trying next...")

    hf_id = HF_FALLBACK.get(backend)
    if hf_id:
        print(f"  no Kaggle Model handle worked -- falling back to HuggingFace '{hf_id}'")
        print(f"  (needs Internet ON; ~61GB for 31B, expect a long download)")
        return hf_id   # from_pretrained resolves HF repo ids directly

    raise RuntimeError(
        f"no handle worked for backend='{backend}'.\n" + "\n".join(errors) +
        "\n  Check the variation slug on the model's Kaggle page and add it to MODEL_HANDLES."
    )
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU",
      "| GPUs:", torch.cuda.device_count() if DEVICE == "cuda" else 0)

# ----------------------------------------------------------------------
# 1. BOOTSTRAP: upgrade transformers so gemma-4's architecture is recognized.
# CONFIRMED (not assumed) via a real Kaggle run: even the latest PyPI release
# (5.13.1) does NOT recognize the `gemma4_unified` architecture -- it's too new
# for any released version. transformers' own error message says to install
# from source instead, so that's what this does. This is almost certainly what
# the teammate's wheelhouse actually contained (a source build, not a PyPI
# release) -- his ">=5.13" comment undersold what was actually needed.
# ----------------------------------------------------------------------
def bootstrap_transformers():
    hits = glob.glob("/kaggle/input/**/wheelhouse", recursive=True)
    if hits:
        wheeldir = hits[0]
        print("wheelhouse found at:", wheeldir)
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-index", "--find-links", wheeldir,
             "-U", "transformers", "accelerate", "bitsandbytes"],
            capture_output=True, text=True, timeout=300,
        )
        print(r.stdout[-1500:])
        if r.returncode != 0:
            print("STDERR:", r.stderr[-1500:])
        return

    print("  no local wheelhouse -- trying a plain PyPI upgrade first...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "transformers", "accelerate", "bitsandbytes"],
        capture_output=True, text=True, timeout=600,
    )
    print(r.stdout[-800:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-800:])

    # verify gemma4_unified is actually registered, not just a high version number --
    # confirmed empirically that a high version number alone isn't sufficient
    check = subprocess.run(
        [sys.executable, "-c",
         "from transformers.models.auto.configuration_auto import CONFIG_MAPPING; "
         "import sys; sys.exit(0 if 'gemma4_unified' in CONFIG_MAPPING else 1)"],
        capture_output=True, text=True,
    )
    if check.returncode == 0:
        print("  gemma4_unified recognized by the PyPI release -- no source build needed")
        return

    print("  PyPI release doesn't recognize gemma4_unified -- installing from GitHub "
          "source instead (this is what transformers' own error message recommends "
          "for architectures too new for any released version; expect this to take "
          "a few minutes, it builds from source)...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "git+https://github.com/huggingface/transformers.git",
         "accelerate", "bitsandbytes"],
        capture_output=True, text=True, timeout=900,
    )
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-1500:])

bootstrap_transformers()
import transformers
print("transformers version:", transformers.__version__)
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
if "gemma4_unified" not in CONFIG_MAPPING:
    print("  WARNING: gemma4_unified still not registered after bootstrap -- gemma-4 will "
          "fail to load. RESTART THE KERNEL and rerun from this cell (Kaggle needs a fresh "
          "process to pick up a newly-installed transformers build; a live upgrade doesn't "
          "hot-swap an already-imported package).")




## Part 1 — Joggota engine (deterministic rules + form features)

In [ ]:
import math
import re
import numpy as np
from collections import Counter
import pandas as pd

def _bn_words(text):
    """Tokenize Bengali text into words (Bengali unicode + ASCII tokens)."""
    return re.findall(r"[\u0980-\u09FF]+|\w+", str(text).lower())

# ==========================================
# 1. THE FORM ENGINE (Akangkha & Asotti)
# ==========================================

def word_entropy(text):
    """Measures structural randomness. High entropy often correlates with verbose hallucinations."""
    words = str(text).split()
    if len(words) < 2: return 0.0
    counts = Counter(words)
    total = len(words)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())

def char_entropy(text):
    """Character-level entropy."""
    text = str(text)
    if len(text) < 2: return 0.0
    counts = Counter(text)
    total = len(text)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())

def novel_char_ratio(context, response):
    """Detects extrinsic hallucinations by finding characters in response not in context."""
    if pd.isna(context) or str(context).strip() in {"", "[NULL]", "nan", "NaN"}:
        return 0.0 # Only applies to RAG/Context rows
    
    ctx_chars  = set(str(context))
    resp_chars = set(str(response))
    return len(resp_chars - ctx_chars) / max(len(resp_chars), 1)

def response_length_ratio(prompt, response):
    """Hallucinated responses are on average 69% longer in this dataset."""
    p_len = len(str(prompt).strip())
    r_len = len(str(response).strip())
    return r_len / max(p_len, 1)


# ==========================================
# 1b. CONTEXT GROUNDING & RESPONSE QUALITY
# ==========================================

def _ngrams(words, n):
    return set(zip(*[words[i:] for i in range(n)])) if len(words) >= n else set()

def context_containment(context, response):
    """
    Fraction of the response's word bigrams that appear verbatim in the context.
    High score = response text is directly lifted from context (strong faithfulness signal).
    Only meaningful for context rows.
    """
    if pd.isna(context) or str(context).strip() in {"", "[NULL]", "nan", "NaN"}:
        return 0.0
    ctx_words = _bn_words(context)
    resp_words = _bn_words(response)
    resp_bigrams = _ngrams(resp_words, 2)
    if not resp_bigrams:
        return 0.0
    ctx_bigrams = _ngrams(ctx_words, 2)
    return len(resp_bigrams & ctx_bigrams) / len(resp_bigrams)

REFUSAL_PHRASES = [
    "আমি জানি না", "আমার জানা নেই", "দুঃখিত", "ক্ষমা করবেন", "ক্ষমাপ্রার্থী",
    "উত্তর দেওয়া সম্ভব নয়", "নিশ্চিত নই", "বলতে পারছি না", "তথ্য নেই",
]

def resp_is_refusal(response):
    """Flags deflection/refusal responses instead of an actual answer."""
    r = str(response)
    return 1.0 if any(phrase in r for phrase in REFUSAL_PHRASES) else 0.0

def resp_code_switch_ratio(response):
    """Ratio of Latin-alphabet word tokens to total word tokens in the response."""
    words = _bn_words(response)
    if not words:
        return 0.0
    latin = sum(1 for w in words if re.fullmatch(r"[a-z0-9]+", w))
    return latin / len(words)

def resp_repetition_score(response):
    """1 - (unique bigrams / total bigrams). Higher = more internal repetition."""
    words = _bn_words(response)
    if len(words) < 2:
        return 0.0
    bigrams = list(zip(words, words[1:]))
    if not bigrams:
        return 0.0
    return 1.0 - (len(set(bigrams)) / len(bigrams))

def resp_is_question(response):
    """Response deflects by ending with a question mark instead of answering."""
    r = str(response).strip()
    return 1.0 if r.endswith("?") else 0.0


# ==========================================
# 2. DETERMINISTIC JOGGOTA (Rules & Regex)
# ==========================================

_MATH_KEYWORDS = ["সম্ভাবনা", "যোগ", "বিয়োগ", "গুণ", "ভাগ", "সংখ্যা",
                  "সমীকরণ", "ক্ষেত্রফল", "পরিসীমা", "লসাগু", "গসাগু"]

_BN_NUMBER_WORDS = [
    "শূন্য", "এক", "দুই", "তিন", "চার", "পাঁচ", "ছয়", "সাত", "আট", "নয়", "দশ",
    "এগার", "বার", "তের", "চৌদ্দ", "পনের", "ষোল", "সতের", "আঠার", "উনিশ", "বিশ",
    "ত্রিশ", "চল্লিশ", "পঞ্চাশ", "ষাট", "সত্তর", "আশি", "নব্বই",
    "শত", "হাজার", "লক্ষ", "কোটি",
]
_MCQ_OPTIONS = {"ক", "খ", "গ", "ঘ", "ঙ"}

def classify_task(prompt: str) -> str:
    """Classifies the task type to route the validation logic."""
    p = str(prompt).lower()
    if re.search(r'বাগধারা|প্রবাদ|প্রবচন', p): return "idiom"
    if re.search(r'অর্থ|ভাবার্থ|শাব্দিক|সমার্থক|বিপরীত|প্রতিশব্দ', p): return "vocabulary"
    if re.search(r'বানান|শুদ্ধ বানান', p): return "spelling"
    # Prefix match on whitespace tokens, not substring search on the raw string --
    # "ভাগ" (divide) and "যোগ" (add) are short enough to false-positive as substrings
    # inside unrelated words like "বিভাগ" (department) and "প্রতিযোগিতা" (competition).
    # Bengali suffixes attach at the end of a stem, so a real match always starts the token.
    tokens = p.split()
    if any(tok.startswith(kw) for tok in tokens for kw in _MATH_KEYWORDS):
        return "math"
    if re.search(r'অনুবাদ|ইংরেজি|translate', p): return "translation"
    if re.search(r'সমাস|ব্যাকরণ|কারক|বিভক্তি|ধাতু|প্রত্যয়|উপসর্গ', p): return "grammar"
    return "factual"

def deterministic_lexical_joggota(prompt: str, response: str, task_type: str) -> float:
    """
    Applies strict rule-based Joggota for specific task types.
    Returns:
      1.0 (Looks Faithful/Passed rule)
      0.0 (Definite Hallucination)
      -1.0 (Abstain - Rule doesn't apply)
    """
    prompt_clean = str(prompt).strip()
    resp_clean = str(response).strip()
    
    if task_type == "spelling":
        # If it's a spelling task, the response should ideally just be the correct word.
        # If the response is extremely long, it's likely hallucinating a whole explanation.
        if len(resp_clean.split()) > 5:
            return 0.0
            
    if task_type == "math":
        # If math, extract numbers. If there are NO numbers in response, it's likely a hallucinated refusal.
        # \d matches Bengali digits (০-৯) too, but misses spelled-out number words
        # ("সাতটি" = "seven") and MCQ letter answers ("গ" = option C) -- both are
        # valid, non-hallucinated responses that don't contain a literal digit.
        resp_numbers = re.findall(r'\d+', resp_clean)
        has_number_word = any(w in resp_clean for w in _BN_NUMBER_WORDS)
        is_mcq_letter = resp_clean.strip('।.?!০-৯0-9') in _MCQ_OPTIONS
        if not resp_numbers and not has_number_word and not is_mcq_letter:
            return 0.0
            
    if task_type == "idiom":
        # Idioms require figurative meaning, not literal.
        COMMON_IDIOMS = {
            "অকাল কুষ্মাণ্ড": {"literal": ["কুমড়া", "সবজি", "ফল"], "figurative": ["অপদার্থ", "অযোগ্য", "বাজে", "অকাজের"]},
            "আকাশ কুসুম": {"literal": ["আকাশ", "ফুল"], "figurative": ["অসম্ভব", "কল্পনা", "অবাস্তব", "মিথ্যা"]},
            "ঘোড়ার ডিম": {"literal": ["ঘোড়া", "ডিম", "অশ্ব"], "figurative": ["অসম্ভব", "অবাস্তব", "কিছুই না", "অস্তিত্বহীন"]},
            "গাছে কাঁঠাল গোঁফে তেল": {"literal": ["গাছ", "কাঁঠাল", "গোঁফ", "তেল"], "figurative": ["আগে", "প্রস্তুতি", "পাওয়ার", "আশায়"]},
            "চোখে সর্ষে ফুল দেখা": {"literal": ["সর্ষে", "ফুল", "চোখ"], "figurative": ["বিপদ", "দিশেহারা", "ঘোরগ্রস্ত", "অন্ধকার"]},
            "হাতের পাঁচ": {"literal": ["হাত", "পাঁচ", "আঙুল"], "figurative": ["সম্বল", "উপায়", "শেষ", "একমাত্র"]},
            "অন্ধের যষ্টি": {"literal": ["অন্ধ", "লাঠি", "যষ্টি"], "figurative": ["অবলম্বন", "একমাত্র", "ভরসা"]},
            "আদা জল খেয়ে লাগা": {"literal": ["আদা", "জল", "পানি"], "figurative": ["উৎসাহে", "চেষ্টা", "প্রাণপণ", "উঠেপড়ে"]},
            "আঙ্গুল ফুলে কলাগাছ": {"literal": ["আঙ্গুল", "কলাগাছ", "গাছ"], "figurative": ["বড়লোক", "উন্নতি", "ধনী", "হঠাৎ"]}
        }
        
        for idiom, keywords in COMMON_IDIOMS.items():
            if idiom in prompt_clean:
                # Check if the LLM took it literally
                has_literal = any(word in resp_clean for word in keywords["literal"])
                has_figurative = any(word in resp_clean for word in keywords["figurative"])
                
                if has_literal and not has_figurative:
                    return 0.0 # Absolute hallucination
                elif has_figurative:
                    return 1.0 # Passed deterministic check
            
    # For vocabulary, we would ideally hook in the local dictionary check here.
    # For now, we abstain if no strict rule was triggered.
    return -1.0 

def cultural_default_penalty(prompt: str, response: str) -> float:
    """
    Detects C1 band 'Cultural Default' hallucinations.
    Returns 1.0 if a known cultural default is detected in the response instead of the expected Bangladesh-specific truth, otherwise 0.0.
    """
    p = str(prompt).lower()
    r = str(response).lower()
    
    # 1. Yunus Nobel (Expected: Peace/শান্তি, Default: Economics/অর্থনীতি)
    if 'ইউনূস' in p or 'yunus' in p:
        if 'নোবেল' in p or 'nobel' in p:
            if 'অর্থনীতি' in r or 'economics' in r:
                return 1.0
                
    # 2. National Poet / Specific Authors (Expected: Nazrul/নজরুল or others, Default: Tagore/রবীন্দ্রনাথ)
    if 'কবি' in p or 'লেখক' in p or 'উপন্যাস' in p or 'কবিতা' in p:
        # If the prompt does NOT mention Tagore, but response DOES
        if 'রবীন্দ্রনাথ' not in p and 'রবীন্দ্রনাথ' in r:
            return 1.0
            
    # 3. ORS / Saline (Expected: Bangladesh/ICDDR,B/Rafiqul, Default: Western/পাশ্চাত্য/WHO)
    if 'স্যালাইন' in p or 'ors' in p or 'saline' in p:
        if 'পাশ্চাত্য' in r or 'western' in r or 'america' in r or 'মার্কিন' in r:
            return 1.0
            
    # 4. First President/PM (Expected: Mujib/Tajuddin, Default: Zia/Others depending on context)
    # This is trickier, keeping it simple for now.
    
    return 0.0

def extract_joggota_features(df: pd.DataFrame) -> pd.DataFrame:
    """Applies the Form Engine and Deterministic Joggota to the entire dataframe."""
    df_out = df.copy()
    
    # Clean NaNs
    df_out['context'] = df_out['context'].fillna('[NULL]')
    df_out['prompt_bn'] = df_out['prompt_bn'].astype(str)
    df_out['response_bn'] = df_out['response_bn'].astype(str)
    
    # 1. Task Classification
    df_out['task_type'] = df_out['prompt_bn'].apply(classify_task)
    
    # 2. Form Engine Features
    df_out['word_entropy'] = df_out['response_bn'].apply(word_entropy)
    df_out['char_entropy'] = df_out['response_bn'].apply(char_entropy)
    df_out['novel_char_ratio'] = df_out.apply(lambda row: novel_char_ratio(row['context'], row['response_bn']), axis=1)
    df_out['length_ratio'] = df_out.apply(lambda row: response_length_ratio(row['prompt_bn'], row['response_bn']), axis=1)
    df_out['resp_len'] = df_out['response_bn'].str.len()

    # 2b. Context Grounding & Response Quality
    df_out['context_containment'] = df_out.apply(lambda row: context_containment(row['context'], row['response_bn']), axis=1)
    df_out['resp_is_refusal'] = df_out['response_bn'].apply(resp_is_refusal)
    df_out['resp_code_switch_ratio'] = df_out['response_bn'].apply(resp_code_switch_ratio)
    df_out['resp_repetition_score'] = df_out['response_bn'].apply(resp_repetition_score)
    df_out['resp_is_question'] = df_out['response_bn'].apply(resp_is_question)

    # 3. Deterministic Joggota
    df_out['deterministic_joggota'] = df_out.apply(
        lambda row: deterministic_lexical_joggota(row['prompt_bn'], row['response_bn'], row['task_type']), axis=1
    )
    
    # 3b. Cultural Default Flag
    df_out['cultural_default_flag'] = df_out.apply(
        lambda row: cultural_default_penalty(row['prompt_bn'], row['response_bn']), axis=1
    )

    return df_out



## Part 1.5 — ground-truth matcher (5 QA sources, CPU only)

This is what produced 0.843. Auto-downloads NCTB-QA, TyDi QA, IndicQA, BanglaRQA and bangla-mmlu (105,262 pairs) if not already cached.

In [ ]:
"""
Ground-truth-source matcher (see CLAUDE.md "Ground-Truth Source Expansion").

One reusable matcher, not N one-off ones: given a competition (prompt_bn, response_bn),
find the closest question in a pool of real QA datasets (NCTB textbooks, TyDi QA gold
passages) and, if the match is close enough to trust, compare response_bn against the
dataset's own answer. Conservative by design -- near-exact matching only (char-ngram
TF-IDF cosine above a high threshold), not fuzzy semantic search. A false-positive match
injects *wrong* ground truth, which is worse than no signal at all.

Sources wired up:
  - NCTB-QA (ShihabReza/nctb-qa on HF, ungated) -- 5,812 Bengali QA pairs built from real
    NCTB Class 6-9 ICT/Science textbooks. Most aligned with the competition's stated Phase 2
    source domains (NCTB textbooks are named explicitly in some_catches.txt).
  - TyDi QA Bengali gold-passage (local, curious_insight/) -- 2,503 general-knowledge QA
    pairs, zero download cost.

BnMMLU (Alvee's real +0.038 lever) is NOT wired up here -- its HuggingFace dataset repo is
gated ("manual" approval required), confirmed via a direct 401 on the parquet endpoint. Only
GitHub code/scripts are public, no data. Needs either HF access-request approval or Alvee's
own copy/token before it can be added the same way.

Output: two soft features, not a hard override (deliberate departure from Alvee's own
math_override, which fires on 0/299 training rows and directly overrides test rows blind):
  - gt_match_score   -- best cosine similarity to any indexed question (0 if nothing close)
  - gt_agreement     -- token-overlap agreement between response_bn and the matched answer,
                        only computed when gt_match_score clears MATCH_THRESHOLD; 0.5
                        (neutral) otherwise so unmatched rows don't bias a downstream model
                        toward either class.
"""
import os
import re
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

MATCH_THRESHOLD = 0.75  # conservative: near-exact question match only, per the integration plan


def _find_input(basename):
    """Phase-2 offline support: locate a bundled copy of `basename` anywhere under
    /kaggle/input (i.e. attached as a Kaggle Dataset). Returns None off-Kaggle."""
    if not os.path.isdir("/kaggle/input"):
        return None
    import glob as _glob
    hits = _glob.glob(f"/kaggle/input/**/{basename}", recursive=True)
    return hits[0] if hits else None


def _resolve_cache(path):
    """Prefer an existing local file, else a /kaggle/input bundled copy, else the
    original path (whose absence triggers the runtime download -- Phase 1 only)."""
    if os.path.exists(path):
        return path
    bundled = _find_input(os.path.basename(path))
    if bundled:
        print(f"  using bundled offline copy: {bundled}")
        return bundled
    return path

_WS_RE = re.compile(r"\s+")
_PUNCT_RE = re.compile(r"[।!?,;:\"'\(\)\[\]]+")
_BN2EN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
_BN_NUMBER_WORDS = {
    "শূন্য": 0, "এক": 1, "দুই": 2, "তিন": 3, "চার": 4, "পাঁচ": 5, "ছয়": 6, "সাত": 7,
    "আট": 8, "নয়": 9, "দশ": 10, "এগারো": 11, "বারো": 12, "তেরো": 13, "চৌদ্দ": 14,
    "পনেরো": 15, "ষোলো": 16, "সতেরো": 17, "আঠারো": 18, "উনিশ": 19, "বিশ": 20,
    "ত্রিশ": 30, "চল্লিশ": 40, "পঞ্চাশ": 50, "ষাট": 60, "সত্তর": 70, "আশি": 80,
    "নব্বই": 90, "শত": 100, "হাজার": 1000, "লক্ষ": 100000, "কোটি": 10000000,
}


def _extract_numbers(text):
    """Digits (Bengali or Latin) plus a best-effort read of spelled-out Bengali
    number words (e.g. "আঠারশ বত্রিশ" -> 1832), reusing the same word list joggota_core.py
    uses for its math rule. Not a full parser -- good enough to stop numeric answers from
    getting silently downgraded to partial token-overlap credit."""
    if not isinstance(text, str):
        return set()
    latin = text.translate(_BN2EN_DIGITS)
    nums = {m for m in re.findall(r"\d+", latin)}
    total, hit = 0, False
    for tok in bn_tokenize(text):
        if tok in _BN_NUMBER_WORDS:
            v = _BN_NUMBER_WORDS[tok]
            if v >= 100:
                total = (total or 1) * v
            else:
                total += v
            hit = True
    if hit and total:
        nums.add(str(total))
    return nums


def normalize(text):
    if not isinstance(text, str):
        return ""
    t = _PUNCT_RE.sub(" ", text)
    t = _WS_RE.sub(" ", t).strip()
    return t


def bn_tokenize(text):
    return re.findall(r"[ঀ-৿]+|\S+", normalize(text))


NCTB_QA_URL = "https://huggingface.co/datasets/ShihabReza/nctb-qa/resolve/main/data/train-00000-of-00001.parquet"


def load_nctb_qa(path="nctb_qa_train.parquet"):
    path = _resolve_cache(path)
    if not os.path.exists(path):
        # Ungated, public dataset (confirmed via a direct 200 on this resolve URL) --
        # safe to fetch at runtime. Phase 1 permits internet; Phase 2 needs this file
        # pre-downloaded and bundled instead (see CLAUDE.md Key Constraints).
        try:
            import urllib.request
            print(f"  {path} not found locally -- downloading from HuggingFace...")
            urllib.request.urlretrieve(NCTB_QA_URL, path)
        except Exception as e:
            print(f"  NCTB-QA download failed ({e}) -- continuing without this source")
            return pd.DataFrame(columns=["question", "answer", "source_name"])
    df = pd.read_parquet(path)
    df = df[df["language"] == "bn"][["question", "answer"]].dropna()
    df["source_name"] = "nctb_qa"
    return df.reset_index(drop=True)


TYDIQA_URLS = [
    "https://huggingface.co/datasets/google-research-datasets/tydiqa/resolve/main/secondary_task/train-00000-of-00001.parquet",
    "https://huggingface.co/datasets/google-research-datasets/tydiqa/resolve/main/secondary_task/validation-00000-of-00001.parquet",
]


def load_tydiqa(path="curious_insight/Sample Selection for QA/Datasets/tydiqa_goldp_bengali.csv",
                 cache_path="tydiqa_goldp_bengali.parquet"):
    path = _resolve_cache(path)
    cache_path = _resolve_cache(cache_path)
    if os.path.exists(path):
        df = pd.read_csv(path)
        df = df[["question", "answer_text"]].dropna().rename(columns={"answer_text": "answer"})
        df["source_name"] = "tydiqa"
        return df.reset_index(drop=True)

    if os.path.exists(cache_path):
        df = pd.read_parquet(cache_path)
        df["source_name"] = "tydiqa"
        return df.reset_index(drop=True)

    # Neither the original local CSV nor a prior download cache exists -- fetch straight
    # from the upstream TyDi QA "secondary_task" (gold-passage) config, ungated, confirmed
    # via a direct 200. This is the exact source the local CSV was itself built from: train
    # (2,390 Bengali rows) + validation (113 Bengali rows) = 2,503, matching the local CSV's
    # row count exactly. Downloads the full multilingual file (~27MB, all languages) since
    # HF doesn't expose a per-language split for this config, then filters to Bengali only
    # and caches the filtered result so this only happens once.
    try:
        import urllib.request
        print(f"  {path} not found locally -- downloading TyDi QA secondary_task from HuggingFace...")
        parts = []
        for i, url in enumerate(TYDIQA_URLS):
            tmp = f"_tydiqa_tmp_{i}.parquet"
            urllib.request.urlretrieve(url, tmp)
            raw = pd.read_parquet(tmp)
            parts.append(raw[raw["id"].str.startswith("bengali")])
            os.remove(tmp)
        bn = pd.concat(parts, ignore_index=True)
        bn["answer"] = bn["answers"].map(
            lambda a: a["text"][0] if isinstance(a, dict) and len(a.get("text", [])) else None
        )
        df = bn[["question", "answer"]].dropna().reset_index(drop=True)
        df.to_parquet(cache_path)
    except Exception as e:
        print(f"  TyDi QA download failed ({e}) -- continuing without this source")
        return pd.DataFrame(columns=["question", "answer", "source_name"])

    df["source_name"] = "tydiqa"
    return df.reset_index(drop=True)


INDICQA_BN_URL = "https://huggingface.co/datasets/ai4bharat/IndicQA/resolve/main/data/indicqa.bn.json"


def load_indicqa(path="indicqa_bn.json"):
    """IndicQA Bengali (ai4bharat/IndicQA, CC-BY-4.0, ungated -- confirmed 200).
    SQuAD-format nesting: data[].paragraphs[].qas[] with `question` + `answers[].text`."""
    path = _resolve_cache(path)
    if not os.path.exists(path):
        try:
            import urllib.request
            print(f"  {path} not found locally -- downloading IndicQA-bn from HuggingFace...")
            urllib.request.urlretrieve(INDICQA_BN_URL, path)
        except Exception as e:
            print(f"  IndicQA download failed ({e}) -- continuing without this source")
            return pd.DataFrame(columns=["question", "answer", "source_name"])
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    rows = []
    for art in d.get("data", []):
        for para in art.get("paragraphs", []):
            for qa in para.get("qas", []):
                answers = qa.get("answers") or []
                if not answers:
                    continue
                text = str(answers[0].get("text", "")).strip()
                if text:
                    rows.append({"question": qa.get("question", ""), "answer": text})
    df = pd.DataFrame(rows)
    if len(df):
        df["source_name"] = "indicqa"
    return df


BANGLARQA_URL = "https://huggingface.co/datasets/sartajekram/BanglaRQA/resolve/main/Train.json"


def load_banglarqa(path="banglarqa_train.json"):
    """BanglaRQA (sartajekram/BanglaRQA, ungated -- confirmed 200), Bengali Wikipedia passages.
    Nesting: data[].qas[] with `question_text`, `is_answerable`, `question_type`,
    `answers.answer_text[]`.

    Two filters matter here. Unanswerable questions carry empty answers and must be dropped.
    Yes/no ("confirmation") questions are dropped too: an answer of "হ্যাঁ"/"না" cannot
    discriminate a faithful response from a hallucinated one, and would fire agreement=1.0
    on any response that happens to contain the token."""
    path = _resolve_cache(path)
    if not os.path.exists(path):
        try:
            import urllib.request
            print(f"  {path} not found locally -- downloading BanglaRQA from HuggingFace...")
            urllib.request.urlretrieve(BANGLARQA_URL, path)
        except Exception as e:
            print(f"  BanglaRQA download failed ({e}) -- continuing without this source")
            return pd.DataFrame(columns=["question", "answer", "source_name"])
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    rows = []
    for rec in d.get("data", []):
        for qa in rec.get("qas", []):
            if str(qa.get("is_answerable", "0")) != "1":
                continue
            if qa.get("question_type") == "confirmation":
                continue
            texts = (qa.get("answers") or {}).get("answer_text") or []
            text = next((str(t).strip() for t in texts if str(t).strip()), "")
            if text:
                rows.append({"question": qa.get("question_text", ""), "answer": text})
    df = pd.DataFrame(rows)
    if len(df):
        df["source_name"] = "banglarqa"
    return df


BANGLA_MMLU_URLS = [
    "https://huggingface.co/datasets/hishab/bangla-mmlu/resolve/main/data/validation-00000-of-00001.parquet",
    "https://huggingface.co/datasets/hishab/bangla-mmlu/resolve/main/data/test-00000-of-00001.parquet",
]
_MCQ_LETTER_TO_IDX = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}


def load_bangla_mmlu(cache_path="bangla_mmlu.parquet"):
    """hishab/bangla-mmlu -- ~88K Bengali exam MCQs (ungated, confirmed 200).

    This is the single most important source in the pool, and it is here for a measured
    reason: the Wikipedia-derived sources above match 92% of *context* rows but only 1.8%
    of *no-context* rows, while this one is almost exactly the mirror image -- 38.9% of
    no-context rows, 0.8% of context rows. No-context rows are the pipeline's dead segment
    (~0.50 hallu-F1, chance), so this source is the only one aimed at it. It stands in for
    the gated samanjoy2/BnMMLU (401, manual approval) that a teammate used for a real
    +0.038 leaderboard gain -- same exam-bank shape, freely accessible.

    Schema: question + choices[] + answer as a letter, so the letter is resolved back into
    the actual option text before use.
    """
    cache_path = _resolve_cache(cache_path)
    if os.path.exists(cache_path):
        df = pd.read_parquet(cache_path)
        df["source_name"] = "bangla_mmlu"
        return df.reset_index(drop=True)

    try:
        import urllib.request
        print(f"  {cache_path} not found locally -- downloading bangla-mmlu from HuggingFace...")
        parts = []
        for i, url in enumerate(BANGLA_MMLU_URLS):
            tmp = f"_bmmlu_tmp_{i}.parquet"
            urllib.request.urlretrieve(url, tmp)
            parts.append(pd.read_parquet(tmp))
            os.remove(tmp)
        raw = pd.concat(parts, ignore_index=True)

        def _resolve(row):
            choices = list(row["choices"]) if row["choices"] is not None else []
            idx = _MCQ_LETTER_TO_IDX.get(str(row["answer"]).strip().upper())
            if idx is None or idx >= len(choices):
                return None
            return choices[idx]

        raw["answer_text"] = raw.apply(_resolve, axis=1)
        df = raw[["question", "answer_text"]].dropna().rename(columns={"answer_text": "answer"})
        df = df[df["question"].astype(str).str.strip().astype(bool)].reset_index(drop=True)
        df.to_parquet(cache_path)
    except Exception as e:
        print(f"  bangla-mmlu download failed ({e}) -- continuing without this source")
        return pd.DataFrame(columns=["question", "answer", "source_name"])

    df["source_name"] = "bangla_mmlu"
    return df.reset_index(drop=True)


def load_all_sources():
    parts = [load_nctb_qa(), load_tydiqa(), load_indicqa(), load_banglarqa(), load_bangla_mmlu()]
    parts = [p for p in parts if len(p)]
    if not parts:
        print("  WARNING: no ground-truth QA sources available (offline?) -- "
              "matcher features will be neutral (match=0, agreement=0.5).")
        return pd.DataFrame(columns=["question", "answer", "source_name"])
    pool = pd.concat(parts, ignore_index=True)
    pool = pool[pool["question"].map(lambda s: len(normalize(s)) > 0)].reset_index(drop=True)
    return pool


class GroundTruthMatcher:
    def __init__(self, pool=None):
        self.pool = pool if pool is not None else load_all_sources()
        self.empty = len(self.pool) == 0
        if self.empty:
            self.vec = self.index = None
            print("GroundTruthMatcher: EMPTY pool -- gt features will be neutral.")
            return
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
        self.index = self.vec.fit_transform(self.pool["question"].map(normalize))
        print(f"GroundTruthMatcher: indexed {len(self.pool)} QA pairs "
              f"({self.pool['source_name'].value_counts().to_dict()})")

    def _best_match(self, prompts, batch=256):
        """Returns (best_idx, best_score) per prompt, batched to bound memory."""
        n = len(prompts)
        best_idx = np.full(n, -1, dtype=int)
        best_score = np.zeros(n)
        qv = self.vec.transform([normalize(p) for p in prompts])
        for i in range(0, n, batch):
            sims = cosine_similarity(qv[i:i + batch], self.index)
            idx = sims.argmax(axis=1)
            score = sims[np.arange(sims.shape[0]), idx]
            best_idx[i:i + batch] = idx
            best_score[i:i + batch] = score
        return best_idx, best_score

    @staticmethod
    def _agreement(response, answer):
        """Numeric answers first (exact digit match required -- no partial credit for a
        wrong number that happens to share surrounding words, e.g. "৭২০ মিটার" vs the
        correct "৬৭০ মিটার"), then token-overlap agreement: containment (short exam
        answers are often substrings/superstrings of the response), Jaccard as fallback."""
        r_norm, a_norm = normalize(response), normalize(answer)
        if not r_norm or not a_norm:
            return 0.5
        a_nums = _extract_numbers(answer)
        if a_nums:
            r_nums = _extract_numbers(response)
            return 1.0 if (a_nums & r_nums) else 0.0
        if a_norm in r_norm or r_norm in a_norm:
            return 1.0
        rt, at = set(bn_tokenize(response)), set(bn_tokenize(answer))
        if not rt or not at:
            return 0.5
        overlap = len(rt & at) / len(at)  # recall against the known answer's tokens
        return float(overlap)

    def score(self, prompts, responses):
        if getattr(self, "empty", False):
            n = len(prompts)
            return (np.zeros(n), np.full(n, 0.5),
                    np.zeros(n, dtype=bool), np.full(n, -1, dtype=int))
        idx, sim = self._best_match(prompts)
        n = len(prompts)
        match_score = sim.copy()
        agreement = np.full(n, 0.5)
        matched = sim >= MATCH_THRESHOLD
        answers = self.pool["answer"].values
        for i in np.where(matched)[0]:
            agreement[i] = self._agreement(responses[i], answers[idx[i]])
        return match_score, agreement, matched, idx


def compute_gt_features(df, matcher=None):
    matcher = matcher or GroundTruthMatcher()
    prompts = df["prompt_bn"].tolist()
    responses = df["response_bn"].tolist()
    match_score, agreement, matched, idx = matcher.score(prompts, responses)
    out = df.copy()
    out["gt_match_score"] = match_score
    out["gt_agreement"] = agreement
    out["gt_matched"] = matched.astype(int)
    return out, idx





## Part 1.6 — hidden-state probe (hooks + in-fold PCA reducer)

In [ ]:
"""
Hidden-state probing: read the judge model's INTERNAL representations, not just its output.

WHY (the one genuinely good idea from twistedfate.txt):
An LLM frequently encodes "I am fabricating this" in its activations while its output
logits stay confidently wrong. So a judge can score AUC~0.50 on a slice of rows while its
hidden states still separate the classes cleanly. That makes probing the correct FIRST
response to "the judge is blind here" -- it costs one forward pass on a model you already
have loaded, versus downloading a bigger one.

It is also the only signal in this pipeline that survives Phase 2 intact. The 0.843 comes
from the ground-truth matcher reading public answer keys; if Phase 2 sources fresh text,
matcher coverage may crater. Hidden states don't care -- they're a property of the model,
not of dataset overlap.

MEMORY NOTE (this is why hooks, not output_hidden_states=True):
`output_hidden_states=True` returns EVERY layer. For a 12B (48 layers, d=3840) at batch 8
x 2048 tokens that's ~6 GB of activations per batch in fp16 -- it will OOM a T4. Forward
hooks on only the layers we want, slicing to the final token INSIDE the hook, hold ~B*d
floats per layer instead. That's ~120 KB. Four orders of magnitude less.

WHY THE LAST TOKEN:
The judge prompts end asking for a single Yes/No token, and the tokenizer runs with
padding_side='left', so position -1 is a real (non-pad) token for every sequence in the
batch and is exactly where the verdict is formed. No attention-mask arithmetic needed.

LEAKAGE WARNING:
The reducer below (StandardScaler + PCA) MUST be fit inside each CV fold, never on the
full training set. With 299 rows and thousands of raw dimensions, fitting the reduction on
all data before cross-validating leaks the validation rows into the projection and
produces a beautiful OOF number that dies on the leaderboard. That has already happened
twice on this project (threshold tuning, model selection). `HiddenStateReducer` is a
fit/transform object specifically so it can be constructed per-fold.
"""
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


def find_decoder_layers(model):
    """Locate the transformer decoder-layer ModuleList across architectures.

    Gemma-4 is a *ConditionalGeneration wrapper (vision + text), so the text layers sit
    somewhere like model.model.language_model.layers -- and the exact path has moved
    between transformers versions. Rather than hardcode a path that silently breaks on the
    next release, try the known ones and then fall back to 'longest ModuleList of modules
    that look like decoder layers', which is robust to renaming.
    """
    candidate_paths = [
        "model.language_model.layers",
        "language_model.model.layers",
        "model.model.language_model.layers",
        "model.layers",
        "model.model.layers",
        "transformer.h",
    ]
    for path in candidate_paths:
        obj = model
        try:
            for part in path.split("."):
                obj = getattr(obj, part)
            if hasattr(obj, "__len__") and len(obj) > 1:
                print(f"  [probe] decoder layers found at '{path}' ({len(obj)} layers)")
                return obj
        except AttributeError:
            continue

    best, best_len, best_name = None, 0, None
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.ModuleList) and len(module) > best_len:
            child = module[0].__class__.__name__.lower()
            if "layer" in child or "block" in child or "decoder" in child:
                best, best_len, best_name = module, len(module), name
    if best is None:
        raise RuntimeError("could not locate decoder layers for hidden-state probing")
    print(f"  [probe] decoder layers found by scan at '{best_name}' ({best_len} layers)")
    return best


def pick_layers(n_layers, fractions=(0.25, 0.50, 0.75, 0.95)):
    """Mid-to-late layers. Early layers encode syntax; factual/uncertainty structure
    consolidates deeper, and the very last layer is already specialized toward next-token
    prediction -- hence 0.95 rather than 1.0."""
    idx = sorted({min(n_layers - 1, max(0, int(round(f * (n_layers - 1))))) for f in fractions})
    return idx


class HiddenStateCollector:
    """Forward hooks on selected decoder layers; captures the final-token activation only.

    Usage:
        collector = HiddenStateCollector(llm)
        ...                      # run forward passes
        vec = collector.pop()    # (batch, n_layers * hidden_dim) for the last forward pass
        collector.remove()       # ALWAYS call -- hooks otherwise persist and leak memory
    """

    def __init__(self, model, fractions=(0.25, 0.50, 0.75, 0.95)):
        self.layers = find_decoder_layers(model)
        self.layer_idx = pick_layers(len(self.layers), fractions)
        self.buffers = {}
        self.handles = []
        for li in self.layer_idx:
            self.handles.append(self.layers[li].register_forward_hook(self._make_hook(li)))
        print(f"  [probe] hooked layers {self.layer_idx} of {len(self.layers)}")

    def _make_hook(self, li):
        def hook(module, inputs, output):
            hs = output[0] if isinstance(output, (tuple, list)) else output
            # slice to the final position INSIDE the hook -- never materialize (B,T,D)
            self.buffers[li] = hs[:, -1, :].detach().float().cpu().numpy()
        return hook

    def pop(self):
        """Concatenated [layer1 | layer2 | ...] for the most recent forward pass."""
        if not self.buffers:
            return None
        vec = np.concatenate([self.buffers[li] for li in self.layer_idx], axis=1)
        self.buffers.clear()
        return vec

    @property
    def width(self):
        return len(self.layer_idx)

    def remove(self):
        """Detach hooks AND drop the reference to the model's layers.

        Dropping self.layers matters: it is a live reference to the model's ModuleList,
        so keeping it alive means a later `del llm; torch.cuda.empty_cache()` cannot
        actually free the weights -- the collector is still holding the graph. On a 12B
        that's survivable; on a 31B using ~18GB of 30GB it will strand VRAM for whatever
        lane loads next.
        """
        for h in self.handles:
            h.remove()
        self.handles.clear()
        self.buffers.clear()
        self.layers = None


class HiddenStateReducer:
    """StandardScaler -> PCA. Construct and fit ONE PER FOLD (see leakage warning above).

    n_components defaults to 32: with ~240 training rows per fold, anything much larger
    starts fitting fold-specific noise. Capped at n_samples-1 automatically.
    """

    def __init__(self, n_components=32, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = None
        self.pca = None

    def fit(self, X):
        X = np.nan_to_num(np.asarray(X, dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)
        self.scaler = StandardScaler().fit(X)
        Xs = self.scaler.transform(X)
        k = int(min(self.n_components, Xs.shape[0] - 1, Xs.shape[1]))
        self.pca = PCA(n_components=max(1, k), random_state=self.random_state).fit(Xs)
        return self

    def transform(self, X):
        X = np.nan_to_num(np.asarray(X, dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)
        return self.pca.transform(self.scaler.transform(X))

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    @property
    def explained(self):
        return float(self.pca.explained_variance_ratio_.sum()) if self.pca is not None else 0.0



## Part 2 — model classes (Translator / NLIScorer / Embedder definitions only)

In [ ]:
"""
Unified Joggota Submission Pipeline
Merges NLI + XGBoost + Joggota deterministic rules + Small LLM Judge.
Implements cross-lingual consistency & cultural-default detection per implementation_plan.md.

VRAM strategy (Kaggle T4, 16 GB):
  Phase 1 (NLI + Embeddings) → GPU, then explicit .to("cpu") + empty_cache()
  Phase 2 (Small LLM Judge)   → Qwen2.5-1.5B or 3B in FP16, only no-context rows
  Fallback: LaBSE sim_premise_response as proxy if LLM fails to load
"""
import os
import gc
import json
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import xgboost as xgb
import torch
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
)

# Import our custom Joggota rules
# [inlined above]

# ----------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------
TRAIN_PATH = "/kaggle/input/competitions/bengali-hallucination/dataset samples.json"
TEST_PATH = "/kaggle/input/competitions/bengali-hallucination/test set.csv"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/bengali-hallucination/sample submission.csv"
SUBMISSION_OUT = "submission.csv"

EMBED_MODEL_NAME = "sentence-transformers/LaBSE"
NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# bn->en translation, used to build the cross-lingual disagreement signal
# ("model correct in English, wrong in Bengali" — the organizers flagged this
#  as the single strongest signal in the benchmark).
TRANSLATOR_CHECKPOINTS = [
    "/kaggle/input/nllb-200-distilled-600m",
    "offline_models/nllb-200-distilled-600m",
    "facebook/nllb-200-distilled-600M",
]

# Offline checkpoints (Kaggle dataset mounts or local downloads)
LLM_CHECKPOINTS = [
    "/kaggle/input/qwen2.5-1.5b-instruct",          # Kaggle dataset
    "offline_models/qwen-2.5-1.5b-instruct",         # Local download
    "Qwen/Qwen2.5-1.5B-Instruct",                    # HuggingFace (Phase 1 internet)
]

RANDOM_STATE = 42
N_FOLDS = 5

# ----------------------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------------------
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    df["context"] = df["context"].replace("[NULL]", np.nan)
    df["context"] = df["context"].replace("", np.nan)
    df["has_context"] = df["context"].notna().astype(int)
    return df


def load_test_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if "context" not in df.columns:
        df["context"] = np.nan
    df["context"] = df["context"].replace("[NULL]", np.nan)
    df["context"] = df["context"].replace("", np.nan)
    df["has_context"] = df["context"].notna().astype(int)
    return df


# ----------------------------------------------------------------------
# 2. MODELS (Embeddings, NLI, LLM Judge)
# ----------------------------------------------------------------------
def _force_gpu_cleanup():
    """Aggressively clear GPU memory including defragmentation."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        try:
            _dummy = torch.zeros(1, device="cuda")
            del _dummy
        except Exception:
            pass


class Embedder:
    def __init__(self, model_name=EMBED_MODEL_NAME, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(model_name, device=self.device)

    def encode(self, texts):
        clean = []
        for t in texts:
            if t is None or (isinstance(t, float) and np.isnan(t)):
                clean.append("।")
            else:
                t = str(t)
                clean.append(t if t.strip() else "।")
        return self.model.encode(
            clean, batch_size=32, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=True,
        )

    def cosine(self, a_emb, b_emb):
        return np.sum(a_emb * b_emb, axis=1)


class NLIScorer:
    def __init__(self, model_name=NLI_MODEL_NAME, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()

    @torch.no_grad()
    def score_batch(self, premises, hypotheses, batch_size=16):
        premises = [
            "" if p is None or (isinstance(p, float) and np.isnan(p)) else str(p)
            for p in premises
        ]
        hypotheses = [
            "" if h is None or (isinstance(h, float) and np.isnan(h)) else str(h)
            for h in hypotheses
        ]
        premises = [p if p.strip() else "।" for p in premises]
        hypotheses = [h if h.strip() else "।" for h in hypotheses]

        all_probs = []
        for i in range(0, len(premises), batch_size):
            p_batch = premises[i : i + batch_size]
            h_batch = hypotheses[i : i + batch_size]
            inputs = self.tokenizer(
                p_batch, h_batch, truncation=True, padding=True,
                max_length=256, return_tensors="pt",
            ).to(self.device)
            logits = self.model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
        return np.vstack(all_probs)


class Translator:
    """
    NLLB-200-distilled-600M, Bengali -> English.
    Used to get a second, English-language NLI verdict on the same
    (premise, response) pair — the model is generally more reliable in
    English, so disagreement between the bn-verdict and en-verdict is a
    hallucination signal in its own right (cross-lingual inconsistency).
    """
    def __init__(self, checkpoint_path=None, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        model_path = checkpoint_path or "facebook/nllb-200-distilled-600M"
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, src_lang="ben_Beng")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(self.device)
        self.model.eval()
        self.eng_id = self.tokenizer.convert_tokens_to_ids("eng_Latn")

    @torch.no_grad()
    def translate_batch(self, texts, batch_size=16, max_length=200):
        clean = []
        for t in texts:
            if t is None or (isinstance(t, float) and np.isnan(t)):
                clean.append("।")
            else:
                t = str(t)
                clean.append(t if t.strip() else "।")

        outputs = []
        for i in range(0, len(clean), batch_size):
            batch = clean[i : i + batch_size]
            inputs = self.tokenizer(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=max_length,
            ).to(self.device)
            generated = self.model.generate(
                **inputs, forced_bos_token_id=self.eng_id,
                max_length=max_length, num_beams=1,
            )
            outputs.extend(self.tokenizer.batch_decode(generated, skip_special_tokens=True))
        return outputs


class SmallLLMJudge:
    """
    Qwen2.5-1.5B-Instruct (or 3B) factual judge for *no-context* rows.

    Uses logit-based scoring (probability of generating '1'/'0' tokens) instead
    of fragile generation + string matching.  Processes in batches for speed.
    """
    def __init__(self, checkpoint_path=None, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        model_path = checkpoint_path or "Qwen/Qwen2.5-1.5B-Instruct"
        print(f"Loading SmallLLMJudge from {model_path} on {self.device}...")

        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=torch.float16, device_map="auto",
        )
        self.model.eval()

        # Pre-compute token IDs for '1' and '0' (Qwen uses these digit tokens)
        self.ids_1 = self._digit_ids("1")
        self.ids_0 = self._digit_ids("0")
        print(f"  Token IDs for '1': {self.ids_1} | '0': {self.ids_0}")

    def _digit_ids(self, digit):
        """Return all token IDs that decode to the given digit string."""
        ids = set()
        for variant in (digit, " " + digit):
            encoded = self.tokenizer.encode(variant, add_special_tokens=False)
            if encoded:
                ids.add(encoded[-1])
        return list(ids)

    def _build_prompt(self, prompt_bn, response_bn):
        """Build a Bangla fact-check prompt with cultural-default awareness."""
        return (
            "আপনি একজন কঠোর বাংলা তথ্য-যাচাইকারী (fact-checker)।\n"
            "বাংলাদেশ/বাঙালি প্রসঙ্গে সঠিক তথ্য দিন। পশ্চিমা/বৈশ্বিক ডিফল্ট উত্তর এড়িয়ে চলুন।\n"
            f"প্রশ্ন: {prompt_bn}\n"
            f"উত্তর: {response_bn}\n\n"
            "উত্তরটি তথ্যগতভাবে সঠিক ও বিশ্বস্ত কিনা? শুধুমাত্র 1 (সঠিক) বা 0 (ভুল) লিখুন।\n"
            "উত্তর:"
        )

    @torch.no_grad()
    def score_batch(self, prompts_bn, responses_bn, batch_size=8):
        """
        Score a list of (prompt, response) pairs.
        Returns array of P(faithful) in [0, 1] — higher = more likely faithful.
        Uses logit-space probability of token '1' vs '0' (more robust than generation).
        """
        n = len(prompts_bn)
        out = np.full(n, 0.5, dtype=np.float32)  # default: abstain

        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch_texts = []
            for i in range(start, end):
                batch_texts.append(
                    self._build_prompt(
                        str(prompts_bn[i]), str(responses_bn[i])
                    )
                )

            inputs = self.tokenizer(
                batch_texts, return_tensors="pt", padding=True,
                truncation=True, max_length=512,
            ).to(self.device)

            logits = self.model(**inputs).logits  # (batch, seq_len, vocab)
            last_logits = logits[:, -1, :].float()  # (batch, vocab) — final position

            # Log-sum-exp for token groups
            log_p1 = torch.logsumexp(last_logits[:, self.ids_1], dim=1)  # (batch,)
            log_p0 = torch.logsumexp(last_logits[:, self.ids_0], dim=1)  # (batch,)

            # Softmax to get P(token='1' | prompt) — proxy for P(faithful)
            stacked = torch.stack([log_p0, log_p1], dim=1)  # (batch, 2)
            prob_faithful = torch.softmax(stacked, dim=1)[:, 1]  # (batch,)
            out[start:end] = prob_faithful.cpu().numpy()

        return out

    def score_no_context_rows(self, df):
        """
        Score only rows WITHOUT context. Returns array of same length as df.
        Context rows get NaN (will be filled later).
        """
        n = len(df)
        out = np.full(n, np.nan, dtype=np.float32)
        mask = df["has_context"] == 0
        idx_list = df.index[mask].tolist()

        if not idx_list:
            print("  (all rows have context — skipping LLM judge)")
            return out

        print(f"  Scoring {len(idx_list)} no-context rows with Small LLM Judge...")
        prompts = [str(df.loc[i]["prompt_bn"]) for i in idx_list]
        responses = [str(df.loc[i]["response_bn"]) for i in idx_list]
        scores = self.score_batch(prompts, responses, batch_size=8)

        for idx, score in zip(idx_list, scores):
            out[idx] = score

        return out


# ----------------------------------------------------------------------
# 3. CROSS-LINGUAL CONSISTENCY FEATURE
# ----------------------------------------------------------------------
def cross_lingual_consistency(df, embedder):
    """
    Compute a cross-lingual consistency score for each row.

    Strategy (lightweight, no second LLM call):
      For rows with context: LaBSE sim(context_EN, response_BN).
      We don't actually translate — LaBSE already lives in a shared multilingual
      space. Instead, we measure the semantic similarity between the context/prompt
      and the response *in LaBSE's space* as a proxy for consistency.

    For no-context rows: we'll use the LLM judge directly (Phase 2),
    so here we provide a fallback: LaBSE similarity between prompt and response
    as a baseline consistency signal.

    Returns a numpy array of scores (higher = more consistent / less hallucinated).
    """
    print("Computing cross-lingual consistency features...")

    response_col = df["response_bn"].astype(str)
    context_col = df["context"]

    # Encode all responses once
    resp_emb = embedder.encode(response_col.tolist())

    # For context rows: use context as the anchor
    # For no-context rows: use prompt as the anchor
    has_ctx = df["has_context"] == 1

    # Context anchor
    ctx_texts = context_col.fillna("").astype(str).tolist()
    ctx_texts = [t if t.strip() and t.strip() != "[NULL]" else "।" for t in ctx_texts]
    ctx_emb = embedder.encode(ctx_texts)

    # Prompt anchor (for no-context rows)
    prompt_texts = df["prompt_bn"].astype(str).tolist()
    prompt_texts = [t if t.strip() else "।" for t in prompt_texts]
    prompt_emb = embedder.encode(prompt_texts)

    # Combine: use context when available, prompt otherwise
    combined_emb = np.where(
        has_ctx.values[:, None], ctx_emb, prompt_emb,
    )
    xlingual_score = embedder.cosine(combined_emb, resp_emb)

    return xlingual_score


# ----------------------------------------------------------------------
# 4. LEXICAL HELPERS
# ----------------------------------------------------------------------
def bn_tokenize(text):
    if not isinstance(text, str):
        if text is None or (isinstance(text, float) and np.isnan(text)):
            return []
        text = str(text)
    return re.findall(r"[\u0980-\u09FF]+|\S+", text)


def token_overlap_ratio(a, b):
    ta, tb = set(bn_tokenize(a)), set(bn_tokenize(b))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


# ----------------------------------------------------------------------
# 5. UNIFIED FEATURE PIPELINE
# ----------------------------------------------------------------------
def build_base_features(df, embedder, nli_scorer, translator=None):
    """Phase 1: NLI + Embeddings + Joggota deterministic features."""
    print("Extracting Form & Deterministic Joggota features...")
    df = extract_joggota_features(df)

    prompt_col = df["prompt_bn"]
    response_col = df["response_bn"]
    context_col = df["context"]

    # --- NLI (context vs response, or prompt vs response if no context) ---
    print("Running NLI...")
    premise_ctx = context_col.fillna(prompt_col)
    probs_ctx = nli_scorer.score_batch(premise_ctx.tolist(), response_col.tolist())
    df["nli_ctx_entail"] = probs_ctx[:, 0]
    df["nli_ctx_contra"] = probs_ctx[:, 2]

    # --- Cross-lingual NLI verification (translate bn->en, re-verify) ---
    if translator is not None:
        print("Translating premise/response to English for cross-lingual check...")
        premise_en = translator.translate_batch(premise_ctx.tolist())
        response_en = translator.translate_batch(response_col.tolist())
        probs_en = nli_scorer.score_batch(premise_en, response_en)
        df["nli_en_entail"] = probs_en[:, 0]
        df["nli_en_contra"] = probs_en[:, 2]
        # Positive = bn NLI says entailed but en (translated) verdict disagrees —
        # the "correct in English, wrong in Bengali" pattern the organizers flagged.
        df["cross_lingual_disagreement"] = df["nli_ctx_entail"] - df["nli_en_entail"]
    else:
        df["nli_en_entail"] = 0.0
        df["nli_en_contra"] = 0.0
        df["cross_lingual_disagreement"] = 0.0

    # --- LaBSE Embeddings ---
    print("Running Embeddings...")
    ctx_or_prompt_emb = embedder.encode(premise_ctx.tolist())
    resp_emb = embedder.encode(response_col.tolist())
    df["sim_premise_response"] = embedder.cosine(ctx_or_prompt_emb, resp_emb)

    # --- Cross-lingual consistency ---
    df["xlingual_consistency"] = cross_lingual_consistency(df, embedder)

    # --- Lexical overlap ---
    df["token_overlap_ctx_resp"] = [
        token_overlap_ratio(p, r)
        for p, r in zip(premise_ctx, response_col)
    ]

    # --- Feature list ---
    feature_cols = [
        "nli_ctx_entail",
        "nli_ctx_contra",
        "nli_en_entail",
        "nli_en_contra",
        "cross_lingual_disagreement",
        "sim_premise_response",
        "xlingual_consistency",
        "token_overlap_ctx_resp",
        "has_context",
        "word_entropy",
        "char_entropy",
        "novel_char_ratio",
        "length_ratio",
        "deterministic_joggota",
        "cultural_default_flag",
        "context_containment",
        "resp_is_refusal",
        "resp_code_switch_ratio",
        "resp_repetition_score",
        "resp_is_question",
    ]
    df[feature_cols] = df[feature_cols].fillna(0)

    return df, feature_cols


# ----------------------------------------------------------------------
# 6. XGBOOST TRAINING & RUN LOGGING
# ----------------------------------------------------------------------
RUN_LOG_PATH = "run_log.csv"


def _log_dataset_stats(df, name):
    n = len(df)
    n_ctx = int(df["has_context"].sum())
    n_noctx = n - n_ctx
    print(f"\n{'='*60}")
    print(f"📊 {name} — {n} rows ({n_ctx} with context, {n_noctx} without)")
    if "label" in df.columns:
        n_faithful = int(df["label"].sum())
        n_hallu = n - n_faithful
        print(
            f"   Labels: {n_faithful} faithful ({n_faithful/n*100:.1f}%) | "
            f"{n_hallu} hallucinated ({n_hallu/n*100:.1f}%)"
        )
    return n, n_ctx, n_noctx


def _log_task_distribution(df):
    if "task_type" not in df.columns:
        return
    print("\n📂 Task Distribution:")
    for task, count in df["task_type"].value_counts().items():
        print(f"   {task:20s}: {count:4d} ({count/len(df)*100:.1f}%)")


def _log_feature_summary(df, feature_cols):
    print("\n📈 Feature Summary (mean ± std across train+test):")
    for col in feature_cols:
        if col in df.columns:
            vals = df[col].dropna()
            print(f"   {col:30s}: {vals.mean():.4f} ± {vals.std():.4f}")


def _log_final_distribution(preds, test_df):
    total = len(preds)
    faithful = int(preds.sum())
    hallu = total - faithful
    print(
        f"\n🎯 Final Predictions: {faithful} faithful ({faithful/total*100:.1f}%) | "
        f"{hallu} hallucinated ({hallu/total*100:.1f}%)"
    )
    print(f"   (out of {total} test rows)")


def _save_run_log(train_df, test_df, feature_cols, cv_results=None, used_fallback=False):
    if "label" not in train_df.columns:
        return
    row = {
        "timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "train_rows": len(train_df),
        "train_faithful": int(train_df["label"].sum()),
        "train_hallucinated": int((1 - train_df["label"]).sum()),
        "train_context": int(train_df["has_context"].sum()),
        "train_no_context": int((1 - train_df["has_context"]).sum()),
        "test_rows": len(test_df),
        "test_context": int(test_df["has_context"].sum()),
        "test_no_context": int((1 - test_df["has_context"]).sum()),
        "used_fallback": int(used_fallback),
    }
    for col in feature_cols:
        if col in train_df.columns:
            row[f"train_{col}_mean"] = float(train_df[col].mean())
            row[f"train_{col}_std"] = float(train_df[col].std())
    if cv_results is not None:
        row["cv_f1_mean"] = float(np.mean(cv_results))
        row["cv_f1_std"] = float(np.std(cv_results))

    log_df = pd.DataFrame([row])
    try:
        existing = pd.read_csv(RUN_LOG_PATH)
        log_df = pd.concat([existing, log_df], ignore_index=True)
    except FileNotFoundError:
        pass
    log_df.to_csv(RUN_LOG_PATH, index=False)
    print(f"\n📝 Run logged to {RUN_LOG_PATH}")


def _cross_validate(X, y, n_folds=5):
    """
    Reports three F1 variants because the competition's own materials disagree:
    Overview says "macro-F1", Rules Section 7 says "Primary metric: binary F1
    on the HALLUCINATED class (label = 0)". label=1 is faithful in this dataset,
    so sklearn's default f1_score(y, pred) (pos_label=1) silently measures the
    FAITHFUL class, not hallucinated — that was a live bug in every prior CV run.
    hallu_f1 is treated as primary (it's the one under an explicit "Primary
    metric" heading) until the organizers resolve the discrepancy.
    """
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    hallu_scores, faithful_scores, macro_scores = [], [], []
    oof_probs = np.zeros(len(y), dtype=np.float64)
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        model = xgb.XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
        )
        model.fit(X_tr, y_tr)
        val_probs = model.predict_proba(X_val)[:, 1]
        oof_probs[val_idx] = val_probs
        preds = (val_probs >= 0.5).astype(int)
        f1_hallu = f1_score(y_val, preds, pos_label=0)
        f1_faithful = f1_score(y_val, preds, pos_label=1)
        f1_macro = f1_score(y_val, preds, average="macro")
        hallu_scores.append(f1_hallu)
        faithful_scores.append(f1_faithful)
        macro_scores.append(f1_macro)
        print(f"   Fold {fold + 1}: hallu_F1={f1_hallu:.4f}  faithful_F1={f1_faithful:.4f}  macro_F1={f1_macro:.4f}")
    print(f"   {'='*30}")
    print(f"   CV hallu_F1 (primary, @0.5): {np.mean(hallu_scores):.4f} ± {np.std(hallu_scores):.4f}")
    print(f"   CV faithful_F1 (@0.5):       {np.mean(faithful_scores):.4f} ± {np.std(faithful_scores):.4f}")
    print(f"   CV macro_F1 (@0.5):          {np.mean(macro_scores):.4f} ± {np.std(macro_scores):.4f}")
    return hallu_scores, oof_probs


def _find_best_threshold(y_true, oof_probs, metric="hallu"):
    """
    Grid-search the decision threshold on out-of-fold probabilities to
    maximize the given F1 variant. Using OOF (not in-sample) probs avoids
    tuning the threshold on data the model has already seen.
    """
    best_t, best_score = 0.5, -1.0
    for t in np.arange(0.05, 0.96, 0.01):
        preds = (oof_probs >= t).astype(int)
        if metric == "hallu":
            score = f1_score(y_true, preds, pos_label=0, zero_division=0)
        elif metric == "macro":
            score = f1_score(y_true, preds, average="macro", zero_division=0)
        else:
            score = f1_score(y_true, preds, pos_label=1, zero_division=0)
        if score > best_score:
            best_t, best_score = t, score
    return best_t, best_score


# ----------------------------------------------------------------------
# 7. MAIN PIPELINE
# ----------------------------------------------------------------------
def train_and_predict():
    print("Loading data...")
    train_df = load_json(TRAIN_PATH)
    test_df = load_test_csv(TEST_PATH)

    _log_dataset_stats(train_df, "TRAIN SET")
    _log_dataset_stats(test_df, "TEST SET")

    # ===================================================================
    # PHASE 1: NLI + Embeddings + Joggota (GPU)
    # ===================================================================
    print("\nLoading NLI, Embedding & Translator models...")
    embedder = Embedder()
    nli_scorer = NLIScorer()

    translator = None
    try:
        translator_ckpt = None
        for candidate in TRANSLATOR_CHECKPOINTS:
            if os.path.exists(candidate):
                translator_ckpt = candidate
                print(f"  Found translator checkpoint: {translator_ckpt}")
                break
        translator_ckpt = translator_ckpt or "facebook/nllb-200-distilled-600M"
        translator = Translator(checkpoint_path=translator_ckpt)
    except Exception as e:
        print(f"  Translator failed to load ({e}) — cross-lingual features will be 0.")

    print("\n--- PHASE 1: Base Features ---")
    print("--- Processing Train Set ---")
    train_df, feature_cols = build_base_features(train_df, embedder, nli_scorer, translator)

    print("\n--- Processing Test Set ---")
    test_df, _ = build_base_features(test_df, embedder, nli_scorer, translator)

    _log_task_distribution(train_df)
    _log_task_distribution(test_df)

    # Aggressive GPU cleanup
    print("\nCleaning up Phase 1 GPU memory...")
    embedder.model.to("cpu")
    nli_scorer.model.to("cpu")
    del embedder, nli_scorer
    if translator is not None:
        translator.model.to("cpu")
        del translator
    _force_gpu_cleanup()

    # ===================================================================
    # PHASE 2: Small LLM Judge (no-context rows only)
    # ===================================================================
    used_fallback = False
    print("\n--- PHASE 2: LLM Judge for No-Context Rows ---")

    # --- Determine checkpoint ---
    checkpoint = None
    for candidate in LLM_CHECKPOINTS:
        if os.path.exists(candidate):
            checkpoint = candidate
            print(f"  Found checkpoint: {checkpoint}")
            break
    if checkpoint is None:
        checkpoint = "Qwen/Qwen2.5-1.5B-Instruct"
        print(f"  Using HuggingFace: {checkpoint}")

    try:
        llm_judge = SmallLLMJudge(checkpoint_path=checkpoint)

        train_llm_scores = llm_judge.score_no_context_rows(train_df)
        test_llm_scores = llm_judge.score_no_context_rows(test_df)

        # Fill NaN (context rows) with 0.0 — XGBoost will learn to rely on NLI
        # features for context rows instead
        train_df["llm_judge_score"] = np.nan_to_num(train_llm_scores, nan=0.0)
        test_df["llm_judge_score"] = np.nan_to_num(test_llm_scores, nan=0.0)
        feature_cols.append("llm_judge_score")

        # Print stats for non-zero scores (no-context rows)
        train_nz = train_df["llm_judge_score"][train_df["has_context"] == 0]
        test_nz = test_df["llm_judge_score"][test_df["has_context"] == 0]
        print(f"\n🧠 LLM Judge Score Summary (no-context rows only):")
        print(f"   Train: mean={train_nz.mean():.3f} | n={len(train_nz)}")
        print(f"   Test:  mean={test_nz.mean():.3f} | n={len(test_nz)}")

        del llm_judge
        _force_gpu_cleanup()

    except Exception as e:
        print(f"⚠️  LLM Judge failed: {e}")
        print(f"   → Falling back to sim_premise_response for no-context signal...")
        used_fallback = True

        # Use the already-computed LaBSE similarity as the proxy
        train_df["llm_judge_score"] = train_df["sim_premise_response"]
        test_df["llm_judge_score"] = test_df["sim_premise_response"]
        feature_cols.append("llm_judge_score")

    # ---- Feature summary ----
    combined = pd.concat([train_df, test_df], ignore_index=True)
    _log_feature_summary(combined, feature_cols)

    # ===================================================================
    # PHASE 3: XGBoost Training & Inference
    # ===================================================================
    print("\n--- PHASE 3: XGBoost Training & Cross-Validation ---")
    X_train = train_df[feature_cols].values
    y_train = train_df["label"].values
    X_test = test_df[feature_cols].values

    print("\n📐 5-Fold Cross-Validation:")
    cv_scores, oof_probs = _cross_validate(X_train, y_train, n_folds=N_FOLDS)

    best_threshold, best_oof_hallu_f1 = _find_best_threshold(y_train, oof_probs, metric="hallu")
    print(f"\n🎯 OOF-tuned threshold (maximizing hallucinated-class F1): "
          f"{best_threshold:.2f} -> OOF hallu_F1={best_oof_hallu_f1:.4f}")

    print("\n🏋️  Training final model on full training set...")
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)

    importance = model.feature_importances_
    print("\n🔍 Feature Importance:")
    for col, imp in sorted(zip(feature_cols, importance), key=lambda x: -x[1]):
        print(f"   {col:30s}: {imp:.4f}")

    print("\nPredicting on test set...")
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= best_threshold).astype(int)

    _log_final_distribution(preds, test_df)

    submission = pd.DataFrame({"id": test_df["id"].values, "label": preds})
    submission.to_csv(SUBMISSION_OUT, index=False)
    print(f"\n✅ Saved submission → {SUBMISSION_OUT}")

    _save_run_log(train_df, test_df, feature_cols, cv_scores, used_fallback)

    if used_fallback:
        print(
            "\n📌 Note: LLM Judge unavailable — used sim_premise_response fallback.\n"
            "   Run download_models.py first for offline Kaggle usage."
        )
    print("✅ Run complete.")


# [__main__ guard stripped -- this notebook drives execution itself]



## Part 2.6 — load competition data + define the judge lane

In [ ]:
# ----------------------------------------------------------------------
# 2. DATA DISCOVERY (adapted from phase2-final.ipynb -- more robust than ours)
# ----------------------------------------------------------------------
NULLISH = {"", "[null]", "null", "none", "nan", "n/a"}

def clean(s, lim):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    return "" if s.lower() in NULLISH else s[:lim]

def find_data():
    roots = [r for r in ["/kaggle/input/bengali-hallucination",
                          "/kaggle/input/competitions/bengali-hallucination",
                          "/kaggle/input", "."] if os.path.isdir(r)]
    test_df, labeled_df = None, None
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True) + \
                 glob.glob(os.path.join(root, "**", "*.json"), recursive=True):
            if os.path.getsize(p) > 80_000_000:
                continue
            try:
                df = pd.read_csv(p) if p.lower().endswith(".csv") else pd.DataFrame(json.load(open(p, encoding="utf-8")))
            except Exception:
                continue
            cols = {c.lower() for c in df.columns}
            if "response_bn" in cols or "response" in cols:
                if "label" in cols and (test_df is None if False else True):
                    if labeled_df is None or len(df) > len(labeled_df):
                        if "label" in cols:
                            labeled_df = df.copy()
                if "label" not in cols and (test_df is None or len(df) > len(test_df)):
                    test_df = df.copy()
    return test_df, labeled_df

test_raw, labeled_raw = find_data()
assert test_raw is not None, "test set not found"
assert labeled_raw is not None, "labeled sample (dataset samples.json) not found"
print("test:", test_raw.shape, "| labeled:", labeled_raw.shape)

def prep(df):
    d = pd.DataFrame()
    d["id"] = df["id"] if "id" in df.columns else np.arange(len(df))
    d["prompt_bn"] = df["prompt_bn"].map(lambda s: clean(s, CFG["max_prompt_chars"]))
    d["response_bn"] = df["response_bn"].map(lambda s: clean(s, CFG["max_resp_chars"]))
    d["context"] = df["context"].map(lambda s: clean(s, CFG["max_ctx_chars"])) if "context" in df.columns else ""
    d["context"] = d["context"].replace("", np.nan)
    d["has_context"] = d["context"].notna().astype(int)
    return d

test = prep(test_raw)
train = prep(labeled_raw)
train["label"] = labeled_raw["label"].astype(int).values
if CFG.get("smoke_n"):
    _n = int(CFG["smoke_n"])
    test = test.head(_n).reset_index(drop=True)
    print(f"*** SMOKE MODE: test capped to {len(test)} rows; train kept full at {len(train)} ***")
print("train has_context:", train["has_context"].mean().round(3),
      "| test has_context:", test["has_context"].mean().round(3))

BN_SENT = re.compile(r"(?<=[।!?\n])\s+")
def sents(t):
    parts = [s.strip() for s in BN_SENT.split(t) if s.strip()]
    return parts if parts else ([t] if t else [])

# ----------------------------------------------------------------------
# 5. LANE 1c: gemma-4-12b-it bilingual judge (verbatim from teammate)
# ----------------------------------------------------------------------
def run_gemma_judge(train_df, test_df):
    import kagglehub
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    print(f"judge backend = {CFG['model_backend']}")
    gemma_path = resolve_model_path(CFG["model_backend"])
    tok = AutoTokenizer.from_pretrained(gemma_path)
    tok.padding_side = "left"; tok.truncation_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # device_map: "balanced" splits evenly across visible GPUs; "auto" fills GPU0 first and
    # can strand a 31B with one card full and the other idle. Unsloth's own Gemma-4 notebook
    # uses device_map="balanced" with the comment "Use 2x Tesla T4s on Kaggle" -- i.e. this
    # exact 2xT4 configuration is a path they test.
    _multi_gpu = torch.cuda.is_available() and torch.cuda.device_count() > 1
    kw = dict(device_map="balanced" if _multi_gpu else "auto",
              attn_implementation="eager")
    print(f"  device_map={kw['device_map']} (GPUs visible: "
          f"{torch.cuda.device_count() if torch.cuda.is_available() else 0})")
    if is_prequantized(gemma_path):
        # Already 4-bit (e.g. an Unsloth build) -- transformers reads the checkpoint's own
        # quantization_config. Passing another BitsAndBytesConfig here conflicts with it.
        print("  checkpoint is pre-quantized -- using its own quantization config")
    else:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)
    try:
        llm = AutoModelForCausalLM.from_pretrained(gemma_path, dtype=torch.float16, **kw).eval()
    except TypeError:
        llm = AutoModelForCausalLM.from_pretrained(gemma_path, torch_dtype=torch.float16, **kw).eval()
    print("loaded gemma-4-12b-it, 4-bit")

    def token_set(variants):
        ids = []
        for v in variants:
            e = tok.encode(v, add_special_tokens=False)
            if len(e) == 1 and e[0] not in ids:
                ids.append(e[0])
        return ids

    YES = token_set(["Yes", "yes", " Yes", " yes", "হ্যাঁ"])
    NO = token_set(["No", "no", " No", " no", "না"])

    def chat(text):
        msgs = [{"role": "user", "content": text}]
        try:
            return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def prompt_A(p, r, c):
        ctx = f"\n\nতথ্যসূত্র (এটিই একমাত্র সত্যের ভিত্তি):\n{c}" if c else ""
        src = "প্রদত্ত তথ্যসূত্র" if c else "বাস্তব, যাচাইযোগ্য জ্ঞান"
        return chat(f"""তুমি একজন কঠোর বাংলা ফ্যাক্ট-চেকার।{ctx}

প্রশ্ন/নির্দেশ:
{p}

মডেলের উত্তর:
{r}

উপরের উত্তরটি কি {src}-এর সাথে সম্পূর্ণ সঙ্গতিপূর্ণ — কোনো ভুল তথ্য, বানোয়াট নাম/তারিখ/সংখ্যা বা অসমর্থিত দাবি নেই? শুধু এক শব্দে উত্তর দাও: Yes অথবা No।""")

    def prompt_B(p, r, c):
        ctx = f"\n\nSOURCE (the only ground truth):\n{c}" if c else ""
        tail = "not supported by the SOURCE" if c else "false in the real world"
        return chat(f"""You are a strict fact-checker for Bengali LLM outputs.{ctx}

PROMPT:
{p}

RESPONSE:
{r}

Does the RESPONSE contain any hallucination — a factual error, a fabricated name/date/number, or a claim {tail}? Answer with exactly one word: Yes or No.""")

    # Hidden-state collector: hooks a few mid-to-late decoder layers and grabs the
    # final-token activation. Attached once, reused across all judge passes; only the
    # prompt_A (Bengali fact-check) pass actually keeps the output -- see below.
    collector = None
    if CFG["probe_hidden_states"]:
        try:
            collector = HiddenStateCollector(llm, fractions=CFG["probe_fractions"])
        except Exception as e:
            print(f"  [probe] disabled -- could not attach hooks ({e})")
            collector = None

    @torch.no_grad()
    def judge(df, builder, flip, tag="", collect=False):
        texts = [builder(df["prompt_bn"].iat[i], df["response_bn"].iat[i],
                         df["context"].iat[i] if pd.notna(df["context"].iat[i]) else "")
                for i in range(len(df))]
        lens = np.array([len(x) for x in tok(texts, truncation=True, max_length=CFG["max_len_llm"])["input_ids"]])
        order = np.argsort(lens, kind="stable")
        pair_ids = YES + NO
        ps, i, done = np.zeros(len(texts)), 0, 0
        # Rows are processed in length-sorted order for batching efficiency, so hidden
        # states must be scattered back to ORIGINAL row positions via idx -- not appended.
        H = None
        while i < len(order):
            take = int(min(CFG["llm_batch"], len(order) - i))
            while take > 1 and int(lens[order[i + take - 1]]) * take > CFG["llm_token_budget"]:
                take -= 1
            idx = order[i:i + take]
            while True:
                try:
                    # Clear any partial captures from a previous OOM'd attempt. Without
                    # this, hooks that fired on early layers before the OOM leave buffers
                    # sized for the LARGER batch; the retry writes smaller ones over only
                    # some layers, and pop() then concatenates mismatched shapes -- either
                    # a crash or, worse, silently misaligned features. 31B OOMs far more
                    # often than 12B, so this path actually gets hit.
                    if collector is not None:
                        collector.buffers.clear()
                    enc = tok([texts[k] for k in idx], return_tensors="pt", padding=True,
                              truncation=True, max_length=CFG["max_len_llm"]).to(llm.device)
                    try:
                        out = llm(**enc, use_cache=False, logits_to_keep=1)
                    except TypeError:
                        out = llm(**enc, use_cache=False)
                    break
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    if len(idx) == 1:
                        raise
                    idx = idx[:max(1, len(idx) // 2)]
            logits = out.logits[:, -1, :].float()
            soft = torch.softmax(logits[:, pair_ids], -1).cpu().numpy()
            ps[idx] = soft[:, :len(YES)].sum(axis=1)

            if collect and collector is not None:
                vec = collector.pop()
                if vec is not None:
                    if H is None:
                        H = np.zeros((len(texts), vec.shape[1]), dtype=np.float32)
                    H[idx] = vec           # scatter by ORIGINAL index, not batch order
            elif collector is not None:
                collector.buffers.clear()  # non-collecting pass: drop, don't accumulate

            i += len(idx); done += len(idx)
            if done % 300 < len(idx):
                print(f"  judge[{tag}] {done}/{len(texts)}")
        return (1.0 - ps if flip else ps), H

    # Hidden states are taken from the prompt_A pass only. A and B ask the same question
    # in different languages/polarities, so their activations are largely redundant --
    # collecting both would double the width for little extra signal. A is the Bengali
    # fact-check framing, the one most aligned with the task.
    pA_tr, H_tr = judge(train_df, prompt_A, flip=False, tag="train/A", collect=True)
    pB_tr, _ = judge(train_df, prompt_B, flip=True, tag="train/B")
    pA_te, H_te = judge(test_df, prompt_A, flip=False, tag="test/A", collect=True)
    pB_te, _ = judge(test_df, prompt_B, flip=True, tag="test/B")

    if collector is not None:
        collector.remove()   # detaches hooks AND drops its ref to the model's layers
        del collector        # ...and drop the collector itself, before empty_cache() below
    if H_tr is not None:
        print(f"  [probe] hidden states -- train {H_tr.shape}, test {H_te.shape}")
        np.save("hidden_train.npy", H_tr)
        np.save("hidden_test.npy", H_te)
        # Sidecar metadata: row counts alone can't tell a 12B run from a 31B run, so a
        # stale .npy from an earlier backend would load silently and be fed to the
        # decision layer as if it matched. Part 4 prints this so the mismatch is visible.
        with open("hidden_meta.json", "w") as f:
            json.dump({"backend": CFG["model_backend"], "width": int(H_tr.shape[1]),
                       "n_train": int(H_tr.shape[0]), "n_test": int(H_te.shape[0]),
                       "layers": CFG["probe_fractions"]}, f)
        print("  [probe] saved hidden_train.npy / hidden_test.npy / hidden_meta.json")

    del llm, tok
    gc.collect(); torch.cuda.empty_cache()
    return (pA_tr, pB_tr), (pA_te, pB_te)






## Part 2.7 — judge smoke test (run BEFORE Part 3)

Part 3 runs NLI, math, cross-lingual and the judge in one cell. If the judge fails there you have already burned the full NLI lane on 2,815 rows. This catches a load failure in under a minute.

In [ ]:
# ---------- smoke test: does the judge load and answer sanely on 3 rows? ----------
# Probing is disabled for the smoke test on purpose: it would write a 3-row
# hidden_train.npy, which Part 4 would then have to detect and reject.
_probe_was = CFG["probe_hidden_states"]
CFG["probe_hidden_states"] = False
_smoke = train.head(3).reset_index(drop=True)
(_pA, _pB), _ = run_gemma_judge(_smoke, _smoke)
CFG["probe_hidden_states"] = _probe_was
print('pA (bn judge, P(faithful)):', _pA)
print('pB (en judge, P(faithful)):', _pB)
print('true labels:                ', _smoke['label'].values)
print()
print('If pA/pB are plausible probabilities (not all 0.5, not NaN) and roughly track the '
      'labels, the judge works. Safe to run Part 3.')



In [ ]:
import os
for f in ["hidden_train.npy", "hidden_test.npy", "hidden_meta.json"]:
    if os.path.exists(f):
        os.remove(f); print("deleted", f)
    else:
        print("not present:", f)


## Part 3 — all lanes; writes feature pickles + hidden_{train,test}.npy

In [ ]:
# ----------------------------------------------------------------------
# 3. LANE 1a: SummaC-ZS windowed NLI (teammate's method + upgraded checkpoint).
#    Loaded ONCE and reused for both the native-Bengali check and the
#    translated-English cross-lingual check below -- no point running our
#    weaker single-shot NLI model in parallel when this one is strictly better.
# ----------------------------------------------------------------------
class SummacNLI:
    def __init__(self):
        from transformers import AutoTokenizer, AutoModelForSequenceClassification

        nli_hits = [p for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "mdeberta" in p.lower()]
        nli_id = os.path.dirname(nli_hits[0]) if nli_hits else "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
        print("SummaC NLI model:", nli_id)
        self.tok = AutoTokenizer.from_pretrained(nli_id)
        try:
            self.mod = AutoModelForSequenceClassification.from_pretrained(nli_id, dtype=torch.float16)
        except TypeError:
            self.mod = AutoModelForSequenceClassification.from_pretrained(nli_id, torch_dtype=torch.float16)
        self.mod = self.mod.to(DEVICE).eval()
        lab2id = {v.lower(): k for k, v in self.mod.config.id2label.items()}
        self.E_ID, self.C_ID = lab2id["entailment"], lab2id["contradiction"]

    @torch.no_grad()
    def _nli_batch(self, premises, hyps):
        out = []
        for i in range(0, len(premises), CFG["nli_batch"]):
            enc = self.tok(premises[i:i + CFG["nli_batch"]], hyps[i:i + CFG["nli_batch"]],
                           truncation=True, max_length=512, padding=True, return_tensors="pt").to(DEVICE)
            out.append(torch.softmax(self.mod(**enc).logits.float(), -1).cpu().numpy())
        return np.concatenate(out)

    @staticmethod
    def _ctx_windows(ctx, win, cap=16):
        ss = sents(ctx)
        if len(ss) <= win:
            return [" ".join(ss)] if ss else []
        stride = max(1, win - 1)
        wins = [" ".join(ss[i:i + win]) for i in range(0, len(ss) - win + 1, stride)]
        wins.append(" ".join(ss[-win:]))
        return wins[:cap]

    def score(self, premises, responses, mask=None, tag=""):
        """SummaC-ZS windowed scoring. `premises`/`responses` are full column
        lists; `mask` restricts scoring to rows where the premise is meaningful
        (e.g. has_context) -- unscored rows get NaN."""
        n = len(premises)
        strict = np.full(n, np.nan)
        soft = np.full(n, np.nan)
        pos = np.arange(n) if mask is None else np.where(mask)[0]
        for j, i in enumerate(pos):
            ctx, resp = premises[i], responses[i]
            hyp = sents(resp)[:12] or [resp]
            st, so = [], []
            for win in CFG["win_list"]:
                wins = self._ctx_windows(ctx, win) or [ctx]
                P, H = [], []
                for h in hyp:
                    for w in wins:
                        P.append(w); H.append(h)
                probs = self._nli_batch(P, H).reshape(len(hyp), len(wins), -1)
                ent = probs[:, :, self.E_ID].max(axis=1)
                con = probs[:, :, self.C_ID].max(axis=1)
                st.append(float(ent.min() - con.max()))
                so.append(float((ent - con).mean()))
            strict[i] = float(np.mean(st)); soft[i] = float(np.mean(so))
            if (j + 1) % 300 == 0:
                print(f"  summac[{tag}] {j+1}/{len(pos)}")
        return strict, soft

    def unload(self):
        self.mod = self.mod.to("cpu")
        del self.mod
        gc.collect(); torch.cuda.empty_cache()


def run_summac_nli(train_df, test_df):
    """
    Unlike the teammate's original (context rows only), this scores EVERY row --
    using context when present, falling back to the prompt otherwise (mask=None).
    No-context rows are our identified weak spot (0.50 hallu_F1 vs 0.79 with
    context in tonight's testing); giving them a windowed NLI signal against the
    prompt, instead of leaving them NaN, directly targets that gap.
    """
    nli = SummacNLI()
    premise_tr = train_df["context"].fillna(train_df["prompt_bn"]).tolist()
    premise_te = test_df["context"].fillna(test_df["prompt_bn"]).tolist()
    strict_tr, soft_tr = nli.score(premise_tr, train_df["response_bn"].tolist(), mask=None, tag="train")
    strict_te, soft_te = nli.score(premise_te, test_df["response_bn"].tolist(), mask=None, tag="test")
    return nli, (strict_tr, soft_tr), (strict_te, soft_te)


# ----------------------------------------------------------------------
# 4. LANE 1b: deterministic math-word-problem solver (verbatim from teammate)
# ----------------------------------------------------------------------
BN2EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
DAYS = ["রবিবার", "সোমবার", "মঙ্গলবার", "বুধবার", "বৃহস্পতিবার", "শুক্রবার", "শনিবার"]

def _nums(s):
    s = re.sub(r"(?<=[০-৯]),(?=[০-৯])", "", s)
    return [int(x.translate(BN2EN)) for x in re.findall(r"[০-৯]+", s)]

def _given_num(resp):
    resp = re.sub(r"(?<=[০-৯]),(?=[০-৯])", "", resp)
    m = re.findall(r"[০-৯]+", resp)
    return int(m[0].translate(BN2EN)) if m else None

def _close(a, b, tol=0.5):
    return a is not None and b is not None and abs(a - b) < tol

def solve_row(prompt, response):
    n = _nums(prompt)
    g = _given_num(response)
    if "সপ্তাহের কোন দিন" in prompt or "সপ্তাহের কোন বার" in prompt:
        day_names = [d for d in DAYS if d in prompt]
        if not day_names or not n:
            return None
        idx = (DAYS.index(day_names[0]) + n[0]) % 7
        return int(DAYS[idx] in response)
    if len(n) < 2:
        return None
    if re.search("একা একটি কাজ|একাই.*দিনে সমাপ্ত", prompt):
        a, b = n[0], n[1]
        correct = (a * b) / (a + b) if (a + b) else 0
        return int(_close(correct, g))
    if "ক, খ ও গ" in prompt:
        a, b, c = n[0], n[1], n[2]
        rate = (1 / a if a else 0) + (1 / b if b else 0) + (1 / c if c else 0)
        correct = 1 / rate if rate else 0
        return int(_close(correct, g))
    if "বয়সের অনুপাত" in prompt:
        a, b, total = n[0], n[1], n[2]
        share = min(a, b)
        correct = total * share / (a + b)
        return int(_close(correct, g))
    if "সংখ্যার অনুপাত" in prompt:
        a, b, total = n[0], n[1], n[2]
        correct = total * b / (a + b)
        return int(_close(correct, g))
    if "ক্রয়মূল্য" in prompt or "কেনা হয়েছিল" in prompt:
        cost, pct = n[0], n[1]
        correct = cost * (1 - pct / 100) if "ক্ষতি" in prompt else cost * (1 + pct / 100)
        return int(_close(correct, g))
    if "সরল সুদ" in prompt:
        principal, rate, time = n[0], n[1], n[2]
        correct = principal * rate * time / 100
        return int(_close(correct, g))
    if "অংশীদারের মধ্যে" in prompt:
        total, a, b, c = n[0], n[1], n[2], n[3]
        correct = total * b / (a + b + c)
        return int(_close(correct, g))
    if "একই দিকে দুইটি বাস" in prompt or "সাইকেল আরোহী একই বিন্দু" in prompt:
        s1, s2, t = n[0], n[1], n[2]
        correct = abs(s1 - s2) * t
        return int(_close(correct, g))
    if "দুইটি শহরের মধ্যে দূরত্ব" in prompt:
        dist, s1, s2 = n[0], n[1], n[2]
        correct = dist / (s1 + s2) if (s1 + s2) else 0
        return int(_close(correct, g))
    if "সংকেত বাতি" in prompt or "বাস স্টপেজ থেকে বাস" in prompt:
        from math import gcd
        a, b, c = n[0], n[1], n[2]
        lcm2 = a * b // gcd(a, b)
        correct = lcm2 * c // gcd(lcm2, c)
        return int(_close(correct, g))
    if "মিশ্রণে চিনি" in prompt:
        a, b, total = n[0], n[1], n[2]
        correct = total * b / (a + b)
        return int(_close(correct, g))
    if "প্যানেল গঠন" in prompt or "উপকমিটি গঠন" in prompt:
        from math import comb
        total, r = n[0], n[1]
        correct = comb(total, r)
        return int(_close(correct, g))
    if "রাশির গড়মান" in prompt or "শিক্ষার্থীর গড় নম্বর" in prompt:
        count, avg1, avg2 = n[0], n[1], n[2]
        correct = avg2 * (count + 1) - avg1 * count
        return int(_close(correct, g))
    return None

def math_feature(df):
    out = np.full(len(df), np.nan)
    for i in range(len(df)):
        lbl = solve_row(df["prompt_bn"].iat[i], df["response_bn"].iat[i])
        if lbl is not None:
            out[i] = lbl
    return out


# ----------------------------------------------------------------------
# 6. LANE 2: translation-based cross-lingual check (reuses the SAME SummacNLI
#    instance from Lane 1a -- no point loading our older, weaker NLI model when
#    his checkpoint is already loaded and strictly better) + lexical/rule features
#    that are genuinely ours: context_containment, novel_char_ratio, joggota rules,
#    response-quality heuristics. None of these exist in his pipeline at all.
# ----------------------------------------------------------------------
def run_cross_lingual_lane(nli: SummacNLI, train_df, test_df,
                            nli_native_tr, nli_native_te, translator=None):
    print("Loading translator for cross-lingual check...")
    if translator is None:
        ckpt = next((c for c in TRANSLATOR_CHECKPOINTS if os.path.exists(c)), None)
        if ckpt is None:
            _hits = [p for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
                     if "nllb" in p.lower()]
            ckpt = os.path.dirname(_hits[0]) if _hits else "facebook/nllb-200-distilled-600M"
        translator = Translator(checkpoint_path=ckpt)

    def translated_score(df, tag):
        premise = df["context"].fillna(df["prompt_bn"]).tolist()
        response = df["response_bn"].tolist()
        premise_en = translator.translate_batch(premise)
        response_en = translator.translate_batch(response)
        return nli.score(premise_en, response_en, mask=None, tag=f"{tag}-en")

    strict_en_tr, soft_en_tr = translated_score(train_df, "train")
    strict_en_te, soft_en_te = translated_score(test_df, "test")

    translator.model.to("cpu"); del translator
    gc.collect(); torch.cuda.empty_cache()

    strict_bn_tr, soft_bn_tr = nli_native_tr
    strict_bn_te, soft_bn_te = nli_native_te
    train_df["nli_summac_soft_en"] = soft_en_tr
    test_df["nli_summac_soft_en"] = soft_en_te
    train_df["cross_lingual_disagreement"] = soft_bn_tr - soft_en_tr
    test_df["cross_lingual_disagreement"] = soft_bn_te - soft_en_te
    return train_df, test_df


def bn_tokenize(text):
    if not isinstance(text, str):
        if text is None or (isinstance(text, float) and np.isnan(text)):
            return []
        text = str(text)
    return re.findall(r"[ঀ-৿]+|\S+", text)


def token_overlap_ratio(a, b):
    ta, tb = set(bn_tokenize(a)), set(bn_tokenize(b))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


def run_lexical_rule_features(train_df, test_df):
    print("Extracting joggota deterministic + lexical features...")
    train_df = extract_joggota_features(train_df)
    test_df = extract_joggota_features(test_df)
    for df in (train_df, test_df):
        premise = df["context"].fillna(df["prompt_bn"])
        df["token_overlap_ctx_resp"] = [
            token_overlap_ratio(p, r) for p, r in zip(premise, df["response_bn"])
        ]
    return train_df, test_df


# ----------------------------------------------------------------------
# LANE 4: ground-truth-source matcher (NCTB-QA + TyDi QA gold-passage, see
# ground_truth_matcher.py / CLAUDE.md "Ground-Truth Source Expansion"). CPU-only,
# no GPU needed -- safe to run alongside any other lane.
# ----------------------------------------------------------------------
def run_ground_truth_matcher(train_df, test_df):
    print("Matching against NCTB-QA + TyDi QA ground-truth sources...")
    matcher = GroundTruthMatcher()
    train_df, _ = compute_gt_features(train_df, matcher)
    test_df, _ = compute_gt_features(test_df, matcher)
    print(f"  train matched: {int(train_df['gt_matched'].sum())}/{len(train_df)} | "
          f"test matched: {int(test_df['gt_matched'].sum())}/{len(test_df)}")
    return train_df, test_df


# ----------------------------------------------------------------------
# 7. RUN EVERYTHING
# ----------------------------------------------------------------------
print("\n=== Lane 1a: SummaC windowed NLI (native, all rows -- context or prompt) ===")
nli_model, nli_native_tr, nli_native_te = run_summac_nli(train, test)

print("\n=== Lane 1b: math solver ===")
math_tr = math_feature(train)
math_te = math_feature(test)
print("math matched -- train:", int(np.sum(~np.isnan(math_tr))), "/", len(math_tr),
      "| test:", int(np.sum(~np.isnan(math_te))), "/", len(math_te))

print("\n=== Lane 2: translation-based cross-lingual check (reuses Lane 1a's NLI model) ===")
train, test = run_cross_lingual_lane(nli_model, train, test, nli_native_tr, nli_native_te)
nli_model.unload()

print("\n=== Lane 1c: gemma-4 bilingual judge ===")
(pA_tr, pB_tr), (pA_te, pB_te) = run_gemma_judge(train, test)

print("\n=== Lane 3: lexical + deterministic-rule features (genuinely ours, no overlap with his) ===")
train, test = run_lexical_rule_features(train, test)

print("\n=== Lane 4: ground-truth-source matcher (NCTB-QA + TyDi QA) ===")
train, test = run_ground_truth_matcher(train, test)

# assemble combined feature frame
def assemble(df, pA, pB, nli_native, math_arr):
    out = df.copy()
    strict, soft = nli_native
    out["llm"] = (pA + pB) / 2
    out["pA"] = pA
    out["pB"] = pB
    out["judge_disagreement"] = np.abs(pA - pB)
    out["nli_summac_strict"] = np.nan_to_num(strict, nan=0.0)
    out["nli_summac_soft"] = np.nan_to_num(soft, nan=0.0)
    out["math_override"] = np.nan_to_num(math_arr, nan=0.5)
    return out

train = assemble(train, pA_tr, pB_tr, nli_native_tr, math_tr)
test = assemble(test, pA_te, pB_te, nli_native_te, math_te)

HIS_FEATURES = ["llm", "pA", "pB", "judge_disagreement", "nli_summac_strict",
                "nli_summac_soft", "math_override", "has_context"]
OUR_FEATURES = ["context_containment", "novel_char_ratio", "word_entropy", "char_entropy",
                "token_overlap_ctx_resp", "length_ratio", "deterministic_joggota",
                "cultural_default_flag", "resp_is_refusal", "resp_code_switch_ratio",
                "resp_repetition_score", "resp_is_question",
                "nli_summac_soft_en", "cross_lingual_disagreement"]
MATCHER_FEATURES = ["gt_match_score", "gt_agreement", "gt_matched"]
ALL_FEATURES = HIS_FEATURES + OUR_FEATURES + MATCHER_FEATURES

train[ALL_FEATURES] = train[ALL_FEATURES].fillna(0)
test[ALL_FEATURES] = test[ALL_FEATURES].fillna(0)

train.to_pickle("fusion_train_features.pkl")
test.to_pickle("fusion_test_features.pkl")
print("\nFeature assembly complete. Saved fusion_train_features.pkl / fusion_test_features.pkl")
print(f"his features: {len(HIS_FEATURES)} | our features: {len(OUR_FEATURES)} | "
      f"matcher features: {len(MATCHER_FEATURES)} | combined: {len(ALL_FEATURES)}")



In [ ]:
# EMERGENCY SNAPSHOT (MANUAL cell): run only after a Part 3 exception, before touching
# anything. In a normal top-to-bottom run it just snapshots existing objects -- harmless.
import pickle

g = globals()
names = ["train", "test", "nli_native_tr", "nli_native_te", "math_tr", "math_te",
         "pA_tr", "pB_tr", "pA_te", "pB_te", "H_tr", "H_te"]
saved = {n: g[n] for n in names if g.get(n) is not None}

with open("part3_snapshot.pkl", "wb") as f:
    pickle.dump(saved, f)

for k, v in saved.items():
    print(f"  {k:16s} {getattr(v, 'shape', None) or (len(v) if hasattr(v, '__len__') else '?')}")
print(f"\nsaved {len(saved)} objects -> part3_snapshot.pkl")
print("the expensive NLI/cross-lingual work is on disk. Re-run only the judge lane.")



## Part 4 — OOF evaluation (incl. hidden-state block) + submissions

Prints the unmatched-rows diagnostic first: judge AUC where the matcher is silent, and whether the hidden-state probe recovers signal the judge output misses. That is the number deciding whether a bigger judge is worth anything.

In [ ]:
"""
Final decision layer: matcher + judge + NLI + lexical + HIDDEN-STATE features,
compared honestly by OOF CV, with the hidden-state reduction fit strictly inside folds.

Feature blocks
--------------
  HIS      - teammate's gemma judge + SummaC NLI + math solver     (8 tabular cols)
  OUR      - joggota lexical/rule/cross-lingual features           (14 tabular cols)
  MATCHER  - ground-truth lookup: gt_match_score/agreement/matched (3 tabular cols)
  HIDDEN   - pooled final-token activations from 4 decoder layers  (RAW MATRIX, ~15k wide)

HIDDEN is handled differently from the rest and that difference is the whole point:
it is NOT concatenated up front. It is reduced (StandardScaler -> PCA) using a reducer
FIT ON THE TRAINING FOLD ONLY, then applied to the held-out fold. Fitting the projection
on all 299 rows before cross-validating would leak validation rows into the components
and produce an inflated OOF number. This project has already shipped two such numbers to
the leaderboard (threshold tuning: OOF 0.714 -> real 0.636; model selection: OOF picked
logreg 0.750 -> real 0.692 while gnb 0.734 -> real 0.717). Not a third.

Reference points to beat, all Kaggle-confirmed:
  his-alone + gnb          OOF 0.7336 -> real 0.717
  his+matcher + logreg     OOF 0.8683 -> real 0.843   <-- current best
"""
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

# [inlined above]

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

train = pd.read_pickle("fusion_train_features.pkl")
test = pd.read_pickle("fusion_test_features.pkl")

HIS_FEATURES = ["llm", "pA", "pB", "judge_disagreement", "nli_summac_strict",
                "nli_summac_soft", "math_override", "has_context"]
OUR_FEATURES = ["context_containment", "novel_char_ratio", "word_entropy", "char_entropy",
                "token_overlap_ctx_resp", "length_ratio", "deterministic_joggota",
                "cultural_default_flag", "resp_is_refusal", "resp_code_switch_ratio",
                "resp_repetition_score", "resp_is_question",
                "nli_summac_soft_en", "cross_lingual_disagreement"]
MATCHER_FEATURES = ["gt_match_score", "gt_agreement", "gt_matched"]

y = train["label"].values.astype(int)
RANDOM_STATE = 42
N_FOLDS = 5
N_PCA = 32

# ---- hidden states (optional -- pipeline degrades cleanly without them) ----
H_train = H_test = None
if os.path.exists("hidden_train.npy") and os.path.exists("hidden_test.npy"):
    H_train = np.load("hidden_train.npy")
    H_test = np.load("hidden_test.npy")
    print(f"hidden states loaded: train {H_train.shape}, test {H_test.shape}")
    # Which model produced these? Row counts match across backends, so a stale file from
    # an earlier 12B run would otherwise be fed to a 31B feature set unnoticed.
    if os.path.exists("hidden_meta.json"):
        import json as _json
        _meta = _json.load(open("hidden_meta.json"))
        print(f"  produced by backend='{_meta.get('backend')}' "
              f"(width {_meta.get('width')}, layers {_meta.get('layers')})")
        print(f"  >> if that is not the backend you just ran, DELETE hidden_*.npy and "
              f"rerun Part 3 <<")
    else:
        print("  (no hidden_meta.json -- cannot confirm which backend produced these; "
              "if you switched models, delete hidden_*.npy and rerun Part 3)")
    if len(H_train) != len(train) or len(H_test) != len(test):
        print("  !! row-count mismatch vs feature pickles -- DISABLING hidden states")
        H_train = H_test = None
else:
    print("hidden states not found (probe disabled or judge lane not rerun) -- "
          "evaluating tabular blocks only")


# ======================================================================
# Diagnostic: is the judge blind exactly where the matcher is silent?
# This is the number that decides whether a bigger judge is worth anything.
# ======================================================================
def diagnose_unmatched():
    if "gt_matched" not in train.columns:
        return
    judge = train["llm"].values
    matched = train["gt_matched"].values.astype(int)
    print("\n" + "=" * 66)
    print("DIAGNOSTIC: judge performance where the matcher contributes nothing")
    print("=" * 66)
    for m, name in [(matched == 1, "matched  "), (matched == 0, "UNMATCHED")]:
        n = int(m.sum())
        if n < 5 or len(np.unique(y[m])) < 2:
            print(f"  {name}: n={n} (too few / single-class)")
            continue
        auc = roc_auc_score(y[m], judge[m])
        print(f"  {name}: n={n:3d}  judge AUC={auc:.3f}  hallu_rate={1-y[m].mean():.2f}")
        if H_train is not None:
            # does the model's INTERNAL state separate these rows better than its output?
            # honest split-half check, not a fitted score
            idx = np.where(m)[0]
            half = len(idx) // 2
            red = HiddenStateReducer(n_components=min(16, half - 1)).fit(H_train[idx[:half]])
            probe = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
            probe.fit(red.transform(H_train[idx[:half]]), y[idx[:half]])
            p = probe.predict_proba(red.transform(H_train[idx[half:]]))[:, 1]
            if len(np.unique(y[idx[half:]])) > 1:
                print(f"             hidden-state probe AUC={roc_auc_score(y[idx[half:]], p):.3f} "
                      f"(held-out half)")
    print("  -> judge AUC ~0.50 on UNMATCHED means the judge output is blind there.")
    print("     If the hidden-state probe AUC is meaningfully higher, the signal EXISTS")
    print("     internally and probing recovers it -- no bigger model required.")


diagnose_unmatched()


def get_model(kind):
    if kind == "gnb":
        # var_smoothing raised from the 1e-9 default: gt_agreement piles mass on exactly
        # 0.5 for unmatched rows -> near-zero within-class variance -> the Gaussian
        # likelihood saturates and drowns every other feature (observed: 0.029 hallu-F1).
        return GaussianNB(var_smoothing=1e-3), True
    if kind == "logreg":
        return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), True
    if kind == "xgb":
        return xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                                 subsample=0.8, colsample_bytree=0.8,
                                 random_state=RANDOM_STATE), False
    raise ValueError(kind)


def build_fold(cols, use_hidden, tr_idx, val_idx):
    """Assemble train/val matrices for one fold. Hidden-state reduction is fit HERE,
    on tr_idx only -- that is the entire anti-leakage mechanism."""
    parts_tr, parts_val = [], []
    if cols:
        Xt = train[cols].values.astype(np.float64)
        parts_tr.append(Xt[tr_idx])
        parts_val.append(Xt[val_idx])
    if use_hidden and H_train is not None:
        red = HiddenStateReducer(n_components=N_PCA, random_state=RANDOM_STATE)
        red.fit(H_train[tr_idx])
        parts_tr.append(red.transform(H_train[tr_idx]))
        parts_val.append(red.transform(H_train[val_idx]))
    return np.hstack(parts_tr), np.hstack(parts_val)


def evaluate(cols, model_kind, name, use_hidden=False):
    if use_hidden and H_train is None:
        return None
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(y))
    fold_thresholds, at_half = [], []
    for tr_idx, val_idx in skf.split(np.zeros(len(y)), y):
        X_tr, X_val = build_fold(cols, use_hidden, tr_idx, val_idx)
        model, scale = get_model(model_kind)
        if scale:
            sc = StandardScaler()
            X_tr = sc.fit_transform(X_tr)
            X_val = sc.transform(X_val)
        model.fit(X_tr, y[tr_idx])
        probs = model.predict_proba(X_val)[:, 1]
        oof[val_idx] = probs
        best_t, best_s = 0.5, -1
        for t in np.arange(0.2, 0.81, 0.02):
            s = f1_score(y[val_idx], (probs >= t).astype(int), pos_label=0, zero_division=0)
            if s > best_s:
                best_t, best_s = t, s
        fold_thresholds.append(best_t)
        at_half.append(f1_score(y[val_idx], (probs >= 0.5).astype(int),
                                pos_label=0, zero_division=0))

    thr = float(np.mean(fold_thresholds))
    hallu = f1_score(y, (oof >= thr).astype(int), pos_label=0, zero_division=0)
    macro = f1_score(y, (oof >= thr).astype(int), average="macro", zero_division=0)
    width = (len(cols) if cols else 0) + (N_PCA if use_hidden else 0)
    print(f"{name:34s} n_feat={width:4d}  F1@0.5={np.mean(at_half):.4f}  "
          f"thr={thr:.2f}  hallu_F1={hallu:.4f}  macro={macro:.4f}")
    return oof, thr, hallu, cols, use_hidden, model_kind


CONFIGS = [
    ("his alone",            HIS_FEATURES, False),
    ("his + matcher",        HIS_FEATURES + MATCHER_FEATURES, False),
    ("matcher alone",        MATCHER_FEATURES, False),
    ("combined",             HIS_FEATURES + OUR_FEATURES + MATCHER_FEATURES, False),
    ("hidden alone",         [], True),
    ("his + hidden",         HIS_FEATURES, True),
    ("matcher + hidden",     MATCHER_FEATURES, True),
    ("his + matcher + hidden", HIS_FEATURES + MATCHER_FEATURES, True),
    ("combined + hidden",    HIS_FEATURES + OUR_FEATURES + MATCHER_FEATURES, True),
]

print(f"\nn={len(train)} | his={len(HIS_FEATURES)} our={len(OUR_FEATURES)} "
      f"matcher={len(MATCHER_FEATURES)} hidden={'yes' if H_train is not None else 'no'}\n")

results = {}
for cfg_name, cols, use_hidden in CONFIGS:
    for kind in ["gnb", "logreg", "xgb"]:
        r = evaluate(cols, kind, f"{cfg_name} + {kind}", use_hidden)
        if r is not None:
            results[f"{cfg_name} + {kind}"] = r

best_key = max(results, key=lambda k: results[k][2])
print(f"\nBest by OOF hallu_F1: {best_key} ({results[best_key][2]:.4f})")
print("  NOTE: OOF has overshot the real leaderboard every time -- logreg by 2.5-5.8pp,")
print("  gnb by ~1.7pp. Subtract before believing it.")


def write_submission(key, filename):
    if key not in results:
        print(f"  (skip {filename} -- config '{key}' unavailable)")
        return
    _, thr, oof_f1, cols, use_hidden, kind = results[key]
    parts_tr, parts_te = [], []
    if cols:
        parts_tr.append(train[cols].values.astype(np.float64))
        parts_te.append(test[cols].values.astype(np.float64))
    if use_hidden:
        red = HiddenStateReducer(n_components=N_PCA, random_state=RANDOM_STATE).fit(H_train)
        parts_tr.append(red.transform(H_train))
        parts_te.append(red.transform(H_test))
    X_full, X_test = np.hstack(parts_tr), np.hstack(parts_te)
    model, scale = get_model(kind)
    if scale:
        sc = StandardScaler()
        X_full = sc.fit_transform(X_full)
        X_test = sc.transform(X_test)
    model.fit(X_full, y)
    preds = (model.predict_proba(X_test)[:, 1] >= thr).astype(int)
    pd.DataFrame({"id": test["id"].values, "label": preds}).to_csv(filename, index=False)
    bal = pd.Series(preds).value_counts(normalize=True).round(3).to_dict()
    print(f"Wrote {filename:38s} [{key}] OOF={oof_f1:.4f} thr={thr:.2f} balance={bal}")


print()
write_submission(best_key, "submission_final.csv")
# fixed reference: the exact config that scored a real 0.843 -- always reproduced so a
# regression is immediately visible rather than hidden behind an OOF-picked winner
write_submission("his + matcher + logreg", "submission_his_matcher_logreg_0843.csv")
write_submission("his + matcher + hidden + logreg", "submission_hidden_logreg.csv")

# Canonical submission.csv = the reproducible best config (the 0.867 line),
# NOT the OOF-picked winner (the note above warns OOF overshoots the real LB).
# Falls back to the 0.843 config if hidden states were not produced this run.
_final_key = ("his + matcher + hidden + logreg"
              if "his + matcher + hidden + logreg" in results
              else "his + matcher + logreg")
write_submission(_final_key, "submission.csv")
print(f"\n>>> Canonical submission.csv written from: {_final_key}")

